# NHA Hackathon — Problem Statement 02  
## Radiological Image Based Condition Detection & Report Correlation  


- All pipeline-stage functions are **empty** and contain only **multiline comments** describing what each function must do.
- A final **main runner section** wires the skeleton together, iterates through discovered cases/files/pages, fills the page-level data structure, and prepares final outputs at the end.

The final participant solution is expected to help adjudicators:
1. Read and describe medical images for condition-relevant features.
2. Check internal consistency of written reports.
3. Correlate image findings with report findings.
4. Check claimed condition / package / STG context.
5. Reason about stage-of-care and episode timeline.
6. Produce reviewer-friendly, assistive outputs.

In [1]:
# =========================
# BASIC IMPORTS
# =========================

from __future__ import annotations

import os
import io
import re
import json
import time
import uuid
import base64
from dataclasses import dataclass, field, asdict
from datetime import datetime, timedelta
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from concurrent.futures import ThreadPoolExecutor

import requests
import fitz  # PyMuPDF
from PIL import Image, ImageOps

try:
    from json_repair import repair_json
except ImportError:
    repair_json = None

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

try:
    import pandas as pd
except ImportError:
    pd = None


In [ ]:
# =========================
# CONFIGURATION
# =========================

INPUT_ROOT = Path("./Claims")
OUTPUT_ROOT = Path("./outputs")

SUPPORTED_IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"}
SUPPORTED_PDF_EXTS = {".pdf"}
SUPPORTED_TEXT_REPORT_EXTS = {".txt", ".md", ".json"}

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# ---- Provider switch --------------------------------------------------------
# Flip between cloud Fireworks (default) and the NHA hackathon LLM gateway.
#   - "fireworks": Fireworks REST API (qwen3-vl-8b by default, vision-capable).
#   - "nha":       NHAclient SDK (Gemma 3 12B for vision, Ministral 8B for text).
# Override at the shell:  PROVIDER=nha jupyter lab ...
PROVIDER = os.environ.get("PROVIDER", "nha").lower().strip()  # default "nha" because the upload target is the NHA platform

# ---- Fireworks settings -----------------------------------------------------
FIREWORKS_URL          = os.environ.get("FIREWORKS_URL", "https://api.fireworks.ai/inference/v1/chat/completions")
FIREWORKS_API_KEY      = os.environ.get("FIREWORKS_API_KEY", "")
FIREWORKS_VISION_MODEL = os.environ.get("FIREWORKS_VISION_MODEL", "accounts/ajeya-rao-k-eckusf6m/deployments/euufjyfd")  # qwen3-vl-8b-instruct
FIREWORKS_TEXT_MODEL   = os.environ.get("FIREWORKS_TEXT_MODEL",   FIREWORKS_VISION_MODEL)

# ---- NHAclient settings -----------------------------------------------------
# Supported models: Ministral 3B / Ministral 8B / Nemotron Nano 30B /
# Gemma 3 12B / Gemma 3 4B. Gemma is the only vision-capable family.
NHA_CLIENT_ID     = os.environ.get("NHA_CLIENT_ID",     "1972e885-bad1-4a57-8d41-fbbdaf9e8b6b")
NHA_CLIENT_SECRET = os.environ.get("NHA_CLIENT_SECRET", "4d5b7a11a7f00fe4d159a17195e3b0e46f07cd98")
NHA_VISION_MODEL  = os.environ.get("NHA_VISION_MODEL",  "Gemma 3 12B")
NHA_TEXT_MODEL    = os.environ.get("NHA_TEXT_MODEL",    "Ministral 8B")
NHA_PROBLEM_STATEMENT = int(os.environ.get("NHA_PROBLEM_STATEMENT", "2"))

# ---- Image preprocessing ----------------------------------------------------
MAX_EDGE         = 1280
JPEG_Q           = 90
PDF_DPI          = 220
NUM_PREDICT      = 4096
TEMPERATURE      = 0.05
SUMMARY_TEMP     = 0.1
SUMMARY_PREDICT  = 4096
MAX_VISION_WORKERS = int(os.environ.get("MAX_VISION_WORKERS", "4"))

# ---- Provider validation ----------------------------------------------------
if PROVIDER not in {"fireworks", "nha"}:
    raise RuntimeError(f"Unknown PROVIDER={PROVIDER!r}. Use 'fireworks' or 'nha'.")
if PROVIDER == "fireworks" and not FIREWORKS_API_KEY:
    print("WARNING: PROVIDER=fireworks but FIREWORKS_API_KEY is not set.")
if PROVIDER == "nha" and not (NHA_CLIENT_ID and NHA_CLIENT_SECRET):
    print("WARNING: PROVIDER=nha but NHA_CLIENT_ID / NHA_CLIENT_SECRET are not set.")

print(f"PROVIDER     = {PROVIDER}")
print(f"vision model = {FIREWORKS_VISION_MODEL if PROVIDER == 'fireworks' else NHA_VISION_MODEL}")
print(f"text model   = {FIREWORKS_TEXT_MODEL   if PROVIDER == 'fireworks' else NHA_TEXT_MODEL}")


In [3]:
# =========================
# DIAGNOSTICS
# =========================
# Run this cell BEFORE the runner to verify everything is wired up.

from pathlib import Path

print(f"INPUT_ROOT      = {INPUT_ROOT}  (exists: {INPUT_ROOT.exists()})")
print(f"OUTPUT_ROOT     = {OUTPUT_ROOT} (exists: {OUTPUT_ROOT.exists()})")
print(f"PROVIDER        = {PROVIDER}")
print(f"FIREWORKS_API_KEY set: {bool(FIREWORKS_API_KEY)}")
print(f"NHA creds set:        {bool(NHA_CLIENT_ID and NHA_CLIENT_SECRET)}")

# Discovery sanity check (should report 92 files / 40 cases for the full PS2 dataset)
try:
    _files = discover_case_files(INPUT_ROOT)
    print(f"discovered files: {len(_files)}")
    from collections import defaultdict
    _cases = defaultdict(int)
    for f in _files:
        _cases[(f.package_code, f.case_id)] += 1
    print(f"cases:            {len(_cases)}")
    by_pkg = defaultdict(int)
    for (pkg, _), _ in _cases.items(): by_pkg[pkg] += 1
    for pkg in sorted(by_pkg): print(f"  {pkg}: {by_pkg[pkg]} cases")
except NameError as e:
    print(f"discover_case_files not yet defined - run the helper cells above first ({e})")

# Quick API ping (1 lightweight text call) so credential issues fail loudly here
# rather than silently producing empty results.
try:
    _resp = _call_text_model(
        "Reply with the single JSON object {\"ok\": true}.",
        "Are you reachable?",
    )
    print(f"API ping reply:  {_resp}")
    if not _resp:
        print("  WARNING: empty reply -- credentials likely missing or call failed.")
except NameError:
    print("_call_text_model not yet defined - run cell 10 first")
except Exception as e:
    print(f"API call failed: {type(e).__name__}: {e}")


INPUT_ROOT      = Claims  (exists: True)
OUTPUT_ROOT     = outputs (exists: True)
PROVIDER        = nha
FIREWORKS_API_KEY set: True
NHA creds set:        True
discover_case_files not yet defined - run the helper cells above first (name 'discover_case_files' is not defined)
_call_text_model not yet defined - run cell 10 first


In [4]:

# =========================
# PACKAGE SCHEMAS / OUTPUT TEMPLATES
# =========================

PACKAGE_FOLDER_TO_CODE = {
    "MC011A": "MC011A",
    "MG029A": "MG029A",
    "SG039": "SG039C",
    "SG039C": "SG039C",
    "SU007A": "SU007A",
}

_DEFAULT = "Not assessed / Not seen"

PACKAGE_OUTPUT_FIELDS: Dict[str, List[str]] = {
    "MC011A": [
        "Claim No. ",
        "Package Code",
        "Type of radiological image in pre-auth/ pre-procedure stage \n(mention absent if no image)",
        "Type of radiological image in cclaim submission/ post-procedure stage \n(mention absent if no image)",
        "LMCA",
        "LAD",
        "LCX ",
        "RCA",
        "LMCA_Post",
        "LAD_Post",
        "LCX_Post",
        "RCA_Post",
        "Stent Visualization",
        "Intra: Within report consistency check \n(Consistent/Partially supported/Unsupported)",
        "Reasoning_Pre_Intra",
        "Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)",
        "Reasoning_Pre_Inter",
        "Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)_Post",
        "Reasoning_Post_Inter",
        "Claim and Package Alignment \n(Both pre procedure and post procedure investigations align/ Only pre-procedure investigation aligns/ Only post-procedure investigtaion aligns/ None of the investigations align)",
        "Reasoning_Alignment",
        "Investigation and STG Alignment (Consistent/Partially supported/Unsupported)",
        "Reason for deviation from STG (to be mapped to PPD and CPD section)",
        "Admission Date\n(DD/MM/YYYY)",
        "Pre-procedure investigation date (From written report, or else the radiological image)\n(DD/MM/YYYY)",
        "Post Procedure Investigation Date (From written report, or else the radiological image)\n(DD/MM/YYYY)",
        "Discharge Date\n(DD/MM/YYYY)",
        "Flag for one month older documents for pre-procedure investigations \n(Yes/ No)",
        "Flag before admssion date or after discharge date documents for post-procedure investigations \n(Yes - outside the window/\nNo - within the window)",
        "Comment on image quality/ usability",
        "Summary PDF",
        "Remarks",
    ],
    "MG029A": [
        "Claim No. ",
        "Package_code",
        "Type of radiological image in cclaim submission/ post-procedure stage \n(mention absent if no image)",
        "Lung fields",
        "Left and right CP angles",
        "Left and right hilum",
        "Midline shift",
        "Cardiac size",
        "Intra Report Consistency Check (within report)\n(Consistent/Partially supported/Unsupported)",
        "Reasoning_Intra",
        "Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)",
        "Reasoning_Inter",
        "Claim and Package Alignment \n(Both pre procedure and post procedure investigations align/ Only pre-procedure investigation aligns/ Only post-procedure investigtaion aligns/ None of the investigations align)",
        "Reason for Non alignment",
        "Investigation and STG Alignment \n(Consistent/Partially supported/Unsupported)",
        "Reason for deviation from STG \n(to be mapped to PPD and CPD section)",
        "Admission Date\n(DD/MM/YYYY)",
        "Post Procedure Investigation Date (From written report, or else the radiological image)\n(DD/MM/YYYY)",
        "Discharge Date\n(DD/MM/YYYY)",
        "Flag before admssion date or after discharge date documents for post-procedure investigations \n(Yes - outside the window/\nNo - within the window)",
        "Comment on image quality/ usability",
        "Summary PDF",
    ],
    "SG039C": [
        "Claim No. ",
        "Package Code",
        "Type of radiological image in pre-auth/ pre-procedure stage \n(mention absent if no image)",
        "Liver",
        "Gall bladder",
        "Spleen",
        "Kidneys",
        "Urinary bladder",
        "Prostate/ Uterus",
        "Peritoneal fluid",
        "Intra:Within report consistency check (Consistent/Partially supported/Unsupportred",
        "Reasoning_Intra",
        "Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)",
        "Reason for Partially supported/Unsupported",
        "Claim and Package Alignment \n(Both pre procedure and post procedure investigations align/ Only pre-procedure investigation aligns/ Only post-procedure investigtaion aligns/ None of the investigations align)",
        "Reasoning_Alignment",
        "Investigation and STG Alignment (Consistent/Partially supported/Unsupported)",
        "Reason for deviation from STG (to be mapped to PPD and CPD section)",
        "Admission Date\n(DD/MM/YYYY)",
        "Pre-procedure investigation date\n(DD/MM/YYYY)",
        "Discharge Date\n(DD/MM/YYYY)",
        "Flag for one month older documents for pre-procedure investigations ",
        "Comment on image quality/ usability",
        "Summary PDF",
    ],
    "SU007A": [
        "Claim No. ",
        "Package code",
        "Type of radiological image in pre-auth/ pre-procedure stage \n(mention absent if no image)",
        "Type of radiological image in cclaim submission/ post-procedure stage\n(mention absent if no image)",
        "Pelvicalyceal system",
        "Ureter",
        "Urinary bladder",
        "Stone visualisation",
        "Any radio opaque shadow or calcification in abdomen region",
        "DJ stent/ PCN tube visualised",
        "Intra: Within report consistency check\n (Consistent/Partially supported/Unsupported)",
        "Reasoning_Pre_Intra",
        "Inter: Image vs Written Report Correlation \n (Consistent/Partially supported/Unsupported)",
        "Reasoning_Pre_Inter",
        "Intra: Within report consistency check\n (Consistent/Partially supported/Unsupported)_Post",
        "Reasoning_Post_Intra",
        "Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)_Post",
        "Reasoning_Post_Inter",
        "Claim and Package Alignment \n(Both pre procedure and post procedure investigations align/ Only pre-procedure investigation aligns/ Only post-procedure investigtaion aligns/ None of the investigations align)",
        "Reasoning_Alignment",
        "Investigation and STG Alignment \n(Consistent/Partially supported/Unsupported)",
        "Reason for deviation from STG (to be mapped to PPD and CPD section)",
        "Admission Date\n(DD/MM/YYYY)",
        "Pre-procedure investigation date\n(DD/MM/YYYY)",
        "Post Procedure Investigation Date\n(DD/MM/YYYY)",
        "Discharge Date\n(DD/MM/YYYY)",
        "Flag for one month older documents for pre-procedure investigations ",
        "Flag before admssion date or after discharge date documents for post-procedure investigations \n(Yes - outside the window/\nNo - within the window)",
        "Comment on image quality/ usability",
        "Summary PDF",
        "Remarks",
    ],
}

PACKAGE_PLACEHOLDERS: Dict[str, Dict[str, Any]] = {
    "MC011A": {
        "Type of radiological image in pre-auth/ pre-procedure stage \n(mention absent if no image)": "Absent",
        "Type of radiological image in cclaim submission/ post-procedure stage \n(mention absent if no image)": "Absent",
        "LMCA": _DEFAULT,
        "LAD": _DEFAULT,
        "LCX ": _DEFAULT,
        "RCA": _DEFAULT,
        "LMCA_Post": _DEFAULT,
        "LAD_Post": _DEFAULT,
        "LCX_Post": _DEFAULT,
        "RCA_Post": _DEFAULT,
        "Stent Visualization": _DEFAULT,
        "Intra: Within report consistency check \n(Consistent/Partially supported/Unsupported)": _DEFAULT,
        "Reasoning_Pre_Intra": _DEFAULT,
        "Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)": _DEFAULT,
        "Reasoning_Pre_Inter": _DEFAULT,
        "Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)_Post": _DEFAULT,
        "Reasoning_Post_Inter": _DEFAULT,
        "Claim and Package Alignment \n(Both pre procedure and post procedure investigations align/ Only pre-procedure investigation aligns/ Only post-procedure investigtaion aligns/ None of the investigations align)": _DEFAULT,
        "Reasoning_Alignment": _DEFAULT,
        "Investigation and STG Alignment (Consistent/Partially supported/Unsupported)": _DEFAULT,
        "Reason for deviation from STG (to be mapped to PPD and CPD section)": _DEFAULT,
        "Admission Date\n(DD/MM/YYYY)": "",
        "Pre-procedure investigation date (From written report, or else the radiological image)\n(DD/MM/YYYY)": "",
        "Post Procedure Investigation Date (From written report, or else the radiological image)\n(DD/MM/YYYY)": "",
        "Discharge Date\n(DD/MM/YYYY)": "",
        "Flag for one month older documents for pre-procedure investigations \n(Yes/ No)": "No",
        "Flag before admssion date or after discharge date documents for post-procedure investigations \n(Yes - outside the window/\nNo - within the window)": "No",
        "Comment on image quality/ usability": _DEFAULT,
        "Summary PDF": "",
        "Remarks": "",
    },
    "MG029A": {
        "Type of radiological image in cclaim submission/ post-procedure stage \n(mention absent if no image)": "Absent",
        "Lung fields": _DEFAULT,
        "Left and right CP angles": _DEFAULT,
        "Left and right hilum": _DEFAULT,
        "Midline shift": _DEFAULT,
        "Cardiac size": _DEFAULT,
        "Intra Report Consistency Check (within report)\n(Consistent/Partially supported/Unsupported)": _DEFAULT,
        "Reasoning_Intra": _DEFAULT,
        "Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)": _DEFAULT,
        "Reasoning_Inter": _DEFAULT,
        "Claim and Package Alignment \n(Both pre procedure and post procedure investigations align/ Only pre-procedure investigation aligns/ Only post-procedure investigtaion aligns/ None of the investigations align)": _DEFAULT,
        "Reason for Non alignment": _DEFAULT,
        "Investigation and STG Alignment \n(Consistent/Partially supported/Unsupported)": _DEFAULT,
        "Reason for deviation from STG \n(to be mapped to PPD and CPD section)": _DEFAULT,
        "Admission Date\n(DD/MM/YYYY)": "",
        "Post Procedure Investigation Date (From written report, or else the radiological image)\n(DD/MM/YYYY)": "",
        "Discharge Date\n(DD/MM/YYYY)": "",
        "Flag before admssion date or after discharge date documents for post-procedure investigations \n(Yes - outside the window/\nNo - within the window)": "No",
        "Comment on image quality/ usability": _DEFAULT,
        "Summary PDF": "",
    },
    "SG039C": {
        "Type of radiological image in pre-auth/ pre-procedure stage \n(mention absent if no image)": "Absent",
        "Liver": _DEFAULT,
        "Gall bladder": _DEFAULT,
        "Spleen": _DEFAULT,
        "Kidneys": _DEFAULT,
        "Urinary bladder": _DEFAULT,
        "Prostate/ Uterus": _DEFAULT,
        "Peritoneal fluid": _DEFAULT,
        "Intra:Within report consistency check (Consistent/Partially supported/Unsupportred": _DEFAULT,
        "Reasoning_Intra": _DEFAULT,
        "Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)": _DEFAULT,
        "Reason for Partially supported/Unsupported": _DEFAULT,
        "Claim and Package Alignment \n(Both pre procedure and post procedure investigations align/ Only pre-procedure investigation aligns/ Only post-procedure investigtaion aligns/ None of the investigations align)": _DEFAULT,
        "Reasoning_Alignment": _DEFAULT,
        "Investigation and STG Alignment (Consistent/Partially supported/Unsupported)": _DEFAULT,
        "Reason for deviation from STG (to be mapped to PPD and CPD section)": _DEFAULT,
        "Admission Date\n(DD/MM/YYYY)": "",
        "Pre-procedure investigation date\n(DD/MM/YYYY)": "",
        "Discharge Date\n(DD/MM/YYYY)": "",
        "Flag for one month older documents for pre-procedure investigations ": "No",
        "Comment on image quality/ usability": _DEFAULT,
        "Summary PDF": "",
    },
    "SU007A": {
        "Type of radiological image in pre-auth/ pre-procedure stage \n(mention absent if no image)": "Absent",
        "Type of radiological image in cclaim submission/ post-procedure stage\n(mention absent if no image)": "Absent",
        "Pelvicalyceal system": _DEFAULT,
        "Ureter": _DEFAULT,
        "Urinary bladder": _DEFAULT,
        "Stone visualisation": _DEFAULT,
        "Any radio opaque shadow or calcification in abdomen region": _DEFAULT,
        "DJ stent/ PCN tube visualised": _DEFAULT,
        "Intra: Within report consistency check\n (Consistent/Partially supported/Unsupported)": _DEFAULT,
        "Reasoning_Pre_Intra": _DEFAULT,
        "Inter: Image vs Written Report Correlation \n (Consistent/Partially supported/Unsupported)": _DEFAULT,
        "Reasoning_Pre_Inter": _DEFAULT,
        "Intra: Within report consistency check\n (Consistent/Partially supported/Unsupported)_Post": _DEFAULT,
        "Reasoning_Post_Intra": _DEFAULT,
        "Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)_Post": _DEFAULT,
        "Reasoning_Post_Inter": _DEFAULT,
        "Claim and Package Alignment \n(Both pre procedure and post procedure investigations align/ Only pre-procedure investigation aligns/ Only post-procedure investigtaion aligns/ None of the investigations align)": _DEFAULT,
        "Reasoning_Alignment": _DEFAULT,
        "Investigation and STG Alignment \n(Consistent/Partially supported/Unsupported)": _DEFAULT,
        "Reason for deviation from STG (to be mapped to PPD and CPD section)": _DEFAULT,
        "Admission Date\n(DD/MM/YYYY)": "",
        "Pre-procedure investigation date\n(DD/MM/YYYY)": "",
        "Post Procedure Investigation Date\n(DD/MM/YYYY)": "",
        "Discharge Date\n(DD/MM/YYYY)": "",
        "Flag for one month older documents for pre-procedure investigations ": "No",
        "Flag before admssion date or after discharge date documents for post-procedure investigations \n(Yes - outside the window/\nNo - within the window)": "No",
        "Comment on image quality/ usability": _DEFAULT,
        "Summary PDF": "",
        "Remarks": "",
    },
}

PACKAGE_IMAGE_TYPE_DEFAULTS = {
    "MC011A": {"pre": "Cardiac angiogram (multiple stills)", "post": "Cardiac angiogram (multiple stills)"},
    "MG029A": {"post": "Chest X ray"},
    "SG039C": {"pre": "USG abdomen"},
    "SU007A": {"pre": "IVP", "post": "X ray KUB"},
}

PACKAGE_SECTION_MAP: Dict[str, List[Tuple[str, List[str]]]] = {
    "MC011A": [
        ("Claim details", ["Claim No. ", "Package Code"]),
        ("1: Image classification", [
            "Type of radiological image in pre-auth/ pre-procedure stage \n(mention absent if no image)",
            "Type of radiological image in cclaim submission/ post-procedure stage \n(mention absent if no image)",
        ]),
        ("2a: Pre-procedure Image description", ["LMCA", "LAD", "LCX ", "RCA"]),
        ("2b: Post-procedure Image description", ["LMCA_Post", "LAD_Post", "LCX_Post", "RCA_Post", "Stent Visualization"]),
        ("3a: Pre-procedure validity of reports", [
            "Intra: Within report consistency check \n(Consistent/Partially supported/Unsupported)",
            "Reasoning_Pre_Intra",
            "Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)",
            "Reasoning_Pre_Inter",
        ]),
        ("3a: Post-procedure validity of reports (only inter)", [
            "Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)_Post",
            "Reasoning_Post_Inter",
        ]),
        ("3: Procedure code and STG alignment", [
            "Claim and Package Alignment \n(Both pre procedure and post procedure investigations align/ Only pre-procedure investigation aligns/ Only post-procedure investigtaion aligns/ None of the investigations align)",
            "Reasoning_Alignment",
            "Investigation and STG Alignment (Consistent/Partially supported/Unsupported)",
            "Reason for deviation from STG (to be mapped to PPD and CPD section)",
        ]),
        ("4: Stage awareness", [
            "Admission Date\n(DD/MM/YYYY)",
            "Pre-procedure investigation date (From written report, or else the radiological image)\n(DD/MM/YYYY)",
            "Post Procedure Investigation Date (From written report, or else the radiological image)\n(DD/MM/YYYY)",
            "Discharge Date\n(DD/MM/YYYY)",
            "Flag for one month older documents for pre-procedure investigations \n(Yes/ No)",
            "Flag before admssion date or after discharge date documents for post-procedure investigations \n(Yes - outside the window/\nNo - within the window)",
        ]),
        ("5. Quality", ["Comment on image quality/ usability"]),
    ],
    "MG029A": [
        ("Claim details", ["Claim No. ", "Package_code"]),
        ("1: Image classification", ["Type of radiological image in cclaim submission/ post-procedure stage \n(mention absent if no image)"]),
        ("2b: Post Procedure Image description", ["Lung fields", "Left and right CP angles", "Left and right hilum", "Midline shift", "Cardiac size"]),
        ("3b: Post-procedure validity of reports", [
            "Intra Report Consistency Check (within report)\n(Consistent/Partially supported/Unsupported)",
            "Reasoning_Intra",
            "Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)",
            "Reasoning_Inter",
        ]),
        ("3: Procedure code and STG alignment", [
            "Claim and Package Alignment \n(Both pre procedure and post procedure investigations align/ Only pre-procedure investigation aligns/ Only post-procedure investigtaion aligns/ None of the investigations align)",
            "Reason for Non alignment",
            "Investigation and STG Alignment \n(Consistent/Partially supported/Unsupported)",
            "Reason for deviation from STG \n(to be mapped to PPD and CPD section)",
        ]),
        ("4: Stage awareness", [
            "Admission Date\n(DD/MM/YYYY)",
            "Post Procedure Investigation Date (From written report, or else the radiological image)\n(DD/MM/YYYY)",
            "Discharge Date\n(DD/MM/YYYY)",
            "Flag before admssion date or after discharge date documents for post-procedure investigations \n(Yes - outside the window/\nNo - within the window)",
        ]),
        ("5. Quality", ["Comment on image quality/ usability"]),
    ],
    "SG039C": [
        ("Claim details", ["Claim No. ", "Package Code"]),
        ("1: Image classification", ["Type of radiological image in pre-auth/ pre-procedure stage \n(mention absent if no image)"]),
        ("2a: Pre-procedure Image description", ["Liver", "Gall bladder", "Spleen", "Kidneys", "Urinary bladder", "Prostate/ Uterus", "Peritoneal fluid"]),
        ("3a: Pre-procedure validity of reports", [
            "Intra:Within report consistency check (Consistent/Partially supported/Unsupportred",
            "Reasoning_Intra",
            "Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)",
            "Reason for Partially supported/Unsupported",
        ]),
        ("3: Procedure code and STG alignment", [
            "Claim and Package Alignment \n(Both pre procedure and post procedure investigations align/ Only pre-procedure investigation aligns/ Only post-procedure investigtaion aligns/ None of the investigations align)",
            "Reasoning_Alignment",
            "Investigation and STG Alignment (Consistent/Partially supported/Unsupported)",
            "Reason for deviation from STG (to be mapped to PPD and CPD section)",
        ]),
        ("4: Stage awareness", [
            "Admission Date\n(DD/MM/YYYY)",
            "Pre-procedure investigation date\n(DD/MM/YYYY)",
            "Discharge Date\n(DD/MM/YYYY)",
            "Flag for one month older documents for pre-procedure investigations ",
        ]),
        ("5. Quality", ["Comment on image quality/ usability"]),
    ],
    "SU007A": [
        ("Claim details", ["Claim No. ", "Package code"]),
        ("1: Image classification", [
            "Type of radiological image in pre-auth/ pre-procedure stage \n(mention absent if no image)",
            "Type of radiological image in cclaim submission/ post-procedure stage\n(mention absent if no image)",
        ]),
        ("2a: Pre- Procedure Image description", ["Pelvicalyceal system", "Ureter", "Urinary bladder", "Stone visualisation"]),
        ("2b: Post Procedure Image description", ["Any radio opaque shadow or calcification in abdomen region", "DJ stent/ PCN tube visualised"]),
        ("3a: Pre-procedure validity of reports", [
            "Intra: Within report consistency check\n (Consistent/Partially supported/Unsupported)",
            "Reasoning_Pre_Intra",
            "Inter: Image vs Written Report Correlation \n (Consistent/Partially supported/Unsupported)",
            "Reasoning_Pre_Inter",
        ]),
        ("3b: Post-procedure validity of reports", [
            "Intra: Within report consistency check\n (Consistent/Partially supported/Unsupported)_Post",
            "Reasoning_Post_Intra",
            "Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)_Post",
            "Reasoning_Post_Inter",
        ]),
        ("3: Procedure code and STG alignment", [
            "Claim and Package Alignment \n(Both pre procedure and post procedure investigations align/ Only pre-procedure investigation aligns/ Only post-procedure investigtaion aligns/ None of the investigations align)",
            "Reasoning_Alignment",
            "Investigation and STG Alignment \n(Consistent/Partially supported/Unsupported)",
            "Reason for deviation from STG (to be mapped to PPD and CPD section)",
        ]),
        ("4: Stage awareness", [
            "Admission Date\n(DD/MM/YYYY)",
            "Pre-procedure investigation date\n(DD/MM/YYYY)",
            "Post Procedure Investigation Date\n(DD/MM/YYYY)",
            "Discharge Date\n(DD/MM/YYYY)",
            "Flag for one month older documents for pre-procedure investigations ",
            "Flag before admssion date or after discharge date documents for post-procedure investigations \n(Yes - outside the window/\nNo - within the window)",
        ]),
        ("5. Quality", ["Comment on image quality/ usability"]),
    ],
}

PACKAGE_TRAILING_FIELDS: Dict[str, List[str]] = {
    "MC011A": ["Summary PDF", "Remarks"],
    "MG029A": ["Summary PDF"],
    "SG039C": ["Summary PDF"],
    "SU007A": ["Summary PDF", "Remarks"],
}

PACKAGE_EXPORT_FIELD_RENAMES: Dict[str, Dict[str, str]] = {
    "MC011A": {
        "LMCA_Post": "LMCA",
        "LAD_Post": "LAD",
        "LCX_Post": "LCX",
        "RCA_Post": "RCA",
        "Reasoning_Pre_Intra": "Reasoning",
        "Reasoning_Pre_Inter": "Reasoning (2)",
        "Reasoning_Post_Inter": "Reasoning",
        "Reasoning_Alignment": "Reasoning",
    },
    "MG029A": {
        "Reasoning_Intra": "Reasoning",
        "Reasoning_Inter": "Reasoning (2)",
    },
    "SG039C": {
        "Reasoning_Intra": "Reasoning",
    },
    "SU007A": {
        "Reasoning_Pre_Intra": "Reasoning",
        "Reasoning_Pre_Inter": "Reasoning (2)",
        "Reasoning_Post_Intra": "Reasoning",
        "Reasoning_Post_Inter": "Reasoning (2)",
        "Reasoning_Alignment": "Reasoning",
    },
}

PACKAGE_IMAGE_SECTION_FIELD_LABELS: Dict[str, Dict[str, str]] = {
    "MC011A": {
        "Type of radiological image in pre-auth/ pre-procedure stage \n(mention absent if no image)": "Type of radiological image in pre-auth/ pre-procedure stage  (mention absent if no image)",
        "Type of radiological image in cclaim submission/ post-procedure stage \n(mention absent if no image)": "Type of radiological image in cclaim submission/ post-procedure stage  (mention absent if no image)",
    },
    "MG029A": {
        "Type of radiological image in cclaim submission/ post-procedure stage \n(mention absent if no image)": "Type of radiological image in cclaim submission/ post-procedure stage  (mention absent if no image)",
    },
    "SG039C": {
        "Type of radiological image in pre-auth/ pre-procedure stage \n(mention absent if no image)": "Type of radiological image in pre-auth/ pre-procedure stage  (mention absent if no image)",
    },
    "SU007A": {
        "Type of radiological image in pre-auth/ pre-procedure stage \n(mention absent if no image)": "Type of radiological image in pre-auth/ pre-procedure stage  (mention absent if no image)",
        "Type of radiological image in cclaim submission/ post-procedure stage\n(mention absent if no image)": "Type of radiological image in cclaim submission/ post-procedure stage  (mention absent if no image)",
    },
}

def initialize_package_row(package_code: str, claim_no: str) -> Dict[str, Any]:
    row = {field_name: "" for field_name in PACKAGE_OUTPUT_FIELDS[package_code]}
    row.update(PACKAGE_PLACEHOLDERS[package_code])

    if package_code == "MG029A":
        row["Package_code"] = package_code
        row["Claim No. "] = claim_no
    elif package_code == "SU007A":
        row["Package code"] = package_code
        row["Claim No. "] = claim_no
    else:
        row["Package Code"] = package_code
        row["Claim No. "] = claim_no
    return row


### Example Usage of NHAclient LLM Call

Supports following models:

- Ministral 3B
- Ministral 8B
- Nemotron Nano 30B
- Gemma 3 12B
- Gemma 3 4B

Can be used by Participants anywhere

In [ ]:
# Optional NHAclient hello-world. Skips automatically when prereqs are missing.
# Reads from the globals set in cell 2 (CONFIGURATION) -- so this is consistent
# with what the pipeline actually uses.

import base64
from pathlib import Path

try:
    from nha_client import NHAclient  # type: ignore
    _has_sdk = True
except ImportError:
    _has_sdk = False

_sample = Path("Sample.jpg")

if not _has_sdk:
    print("NHAclient SDK not installed -- skipping demo (this is fine for local Fireworks runs).")
elif not (NHA_CLIENT_ID and NHA_CLIENT_SECRET):
    print("NHA_CLIENT_ID / NHA_CLIENT_SECRET globals are empty -- skipping demo.")
elif not _sample.exists():
    print(f"{_sample} not found -- skipping demo (the pipeline does not need it).")
else:
    nc = NHAclient(NHA_CLIENT_ID, NHA_CLIENT_SECRET)
    image_bytes  = _sample.read_bytes()
    image_base64 = base64.b64encode(image_bytes).decode("utf-8")
    data_url     = f"data:image/jpeg;base64,{image_base64}"
    response = nc.completion(
        model=NHA_VISION_MODEL,
        messages=[{
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": data_url}},
                {"type": "text",      "text": "What do you see in this image? Reply with one short sentence."},
            ],
        }],
        metadata={"problem_statement": NHA_PROBLEM_STATEMENT},
    )
    # Just print the assistant content, not the whole envelope.
    try:
        print(response["choices"][0]["message"]["content"])
    except Exception:
        print(response)


In [6]:

# =========================
# DATA STRUCTURES
# =========================

@dataclass
class DiscoveredFile:
    package_code: str
    case_id: str
    file_path: str
    file_name: str
    file_ext: str
    file_kind: str  # medical_image | pdf_document | written_report | other


@dataclass
class PageData:
    package_code: str
    case_id: str
    source_file: str
    file_kind: str
    page_number: int
    page_id: str

    image_path: Optional[str] = None
    extracted_text: str = ""
    report_text: str = ""
    metadata: Dict[str, Any] = field(default_factory=dict)

    image_quality_assessment: Dict[str, Any] = field(default_factory=dict)
    image_observations: Dict[str, Any] = field(default_factory=dict)
    report_observations: Dict[str, Any] = field(default_factory=dict)
    internal_report_consistency: Dict[str, Any] = field(default_factory=dict)
    image_report_correlation: Dict[str, Any] = field(default_factory=dict)
    package_stg_alignment: Dict[str, Any] = field(default_factory=dict)
    timeline_stage_assessment: Dict[str, Any] = field(default_factory=dict)
    reviewer_notes: Dict[str, Any] = field(default_factory=dict)


@dataclass
class CaseResult:
    package_code: str
    case_id: str
    pages: List[PageData] = field(default_factory=list)
    structured_output_row: Dict[str, Any] = field(default_factory=dict)
    textual_summary: str = ""
    reviewer_flags: List[Dict[str, Any]] = field(default_factory=list)
    final_case_summary: Dict[str, Any] = field(default_factory=dict)


@dataclass
class RunOutputs:
    all_case_results: List[CaseResult] = field(default_factory=list)
    package_json_rows: Dict[str, List[Dict[str, Any]]] = field(default_factory=dict)
    final_reviewer_flags: List[Dict[str, Any]] = field(default_factory=list)
    final_case_summaries: List[Dict[str, Any]] = field(default_factory=list)
    written_files: Dict[str, Any] = field(default_factory=dict)


In [7]:
# =========================
# DATA ONBOARDING HELPERS
# =========================

def classify_input_file(file_path: Path) -> str:
    """Map a file extension to a coarse pipeline kind."""
    ext = file_path.suffix.lower()
    if ext in SUPPORTED_IMAGE_EXTS:
        return "medical_image"
    if ext in SUPPORTED_PDF_EXTS:
        return "pdf_document"
    if ext in SUPPORTED_TEXT_REPORT_EXTS:
        return "written_report"
    return "other"


def discover_case_files(input_root: Path) -> List[DiscoveredFile]:
    """Walk INPUT_ROOT/<package>/<case>/* and emit DiscoveredFile entries."""
    discovered: List[DiscoveredFile] = []
    if not input_root.exists():
        return discovered

    for file_path in sorted([p for p in input_root.rglob("*") if p.is_file()]):
        try:
            rel = file_path.relative_to(input_root)
        except Exception:
            continue
        parts = rel.parts
        if len(parts) < 2:
            continue
        package_folder = parts[0]
        package_code = PACKAGE_FOLDER_TO_CODE.get(package_folder)
        if not package_code or package_code not in PACKAGE_OUTPUT_FIELDS:
            continue

        # Skip notebook-generated artefacts in claim folders
        name = file_path.name
        if name.startswith(".") or name in {
            "image_catalog.json", "image_catalog.md",
            "claim_summary.json", "claim_summary.md",
        }:
            continue

        case_id = parts[1] if len(parts) >= 3 else file_path.stem
        kind = classify_input_file(file_path)
        if kind == "other":
            continue

        discovered.append(DiscoveredFile(
            package_code=package_code,
            case_id=case_id,
            file_path=str(file_path),
            file_name=file_path.name,
            file_ext=file_path.suffix.lower(),
            file_kind=kind,
        ))
    return discovered


def split_file_into_pages(discovered_file: DiscoveredFile) -> List[Dict[str, Any]]:
    """One stub per renderable page. PDFs expand by page; images/text become a single stub."""
    file_path = Path(discovered_file.file_path)
    base = {
        "package_code": discovered_file.package_code,
        "case_id": discovered_file.case_id,
        "source_file": discovered_file.file_name,
        "file_kind": discovered_file.file_kind,
        "file_path": str(file_path),
    }

    if discovered_file.file_kind == "pdf_document":
        try:
            with fitz.open(str(file_path)) as doc:
                num_pages = len(doc)
        except Exception:
            num_pages = 1
        num_pages = max(1, num_pages)
        return [
            {
                **base,
                "page_number": i + 1,
                "page_id": f"{discovered_file.package_code}::{discovered_file.case_id}::{discovered_file.file_name}::{i + 1}",
                "image_path": str(file_path),
            }
            for i in range(num_pages)
        ]

    return [{
        **base,
        "page_number": 1,
        "page_id": f"{discovered_file.package_code}::{discovered_file.case_id}::{discovered_file.file_name}::1",
        "image_path": str(file_path) if discovered_file.file_kind in {"medical_image", "pdf_document"} else None,
    }]


In [8]:
# =========================
# 1. PAGE-LEVEL INGESTION / EXTRACTION
# =========================

# ---- Per-image vision system prompt (verbatim port of ps2_image_catalog) ----
VISION_SYSTEM = r"""You are a medical document OCR + description assistant for India's
AB PM-JAY claims dataset. You receive ONE image at a time. The image may be:
  - A medical scan (X-ray, IVP, KUB, USG, CT, MRI, coronary angiogram still)
  - A typed clinical / radiology report (English or Indic-language)
  - A handwritten doctor's note or prescription
  - A doctor's stamp, signature block, hospital letterhead
  - A photograph of a cath-lab monitor

ROLE - STRICTLY DESCRIPTIVE
- Do not diagnose. Do not approve / reject. Do not recommend treatment.
- Describe what you SEE. Cite uncertainty when a feature is unclear.

OCR + LANGUAGE
- Transcribe every legible line VERBATIM in its original script for the
  "ocr.verbatim" field.
- Detect the language for "ocr.language": one of English, Hindi, Tamil, Telugu,
  Bengali, Marathi, Gujarati, Kannada, Malayalam, Punjabi, Odia, Urdu,
  Assamese, mixed, or none.
- Translate the verbatim text into English for "ocr.english".
- When a date is in DD-MM-YYYY / DD.MM.YYYY / YYYY-MM-DD / Indic numerals,
  normalise to DD/MM/YYYY in "dates_found".

IMAGE_TYPE (pick ONE)
  "Chest X-Ray"   "Abdominal X-Ray (KUB)"   "IVP"   "Urethrogram"
  "Ultrasound"    "CT"   "MRI"   "Coronary Angiogram"
  "Typed report"  "Handwritten note"   "Stamp/Signature"
  "Hospital letterhead"   "Patient photo"   "Other"

OUTPUT - exactly ONE JSON object, no markdown, no code fence:

{
  "image_type":     "<from list>",
  "body_region":    "<string or 'N/A'>",
  "modality_view":  "<string or 'N/A'>",
  "stage_of_care":  "pre-procedure | intra-procedure | post-procedure | uncertain | n/a",
  "stage_evidence": "<short clue>",
  "ocr": {
    "language":  "<language>",
    "verbatim":  "<every legible line in original script>",
    "english":   "<English translation>"
  },
  "summary":          "<1-3 sentences>",
  "key_observations": ["<bullet 1>", "<bullet 2>"],
  "dates_found":      ["<DD/MM/YYYY>"],
  "image_quality":    {
    "rating":      "good | fair | poor",
    "limitations": ["<e.g. blur, low contrast, partial view>"]
  }
}

FORBIDDEN: diagnosing, inventing text, approving claims, prose around the JSON."""


REPORT_IMAGE_TYPES = {"Typed report", "Handwritten note", "Stamp/Signature", "Hospital letterhead"}


# ---- Image loading helpers --------------------------------------------------
def _load_pil_for_page(page_data: PageData) -> Optional[Image.Image]:
    src = page_data.image_path or page_data.metadata.get("file_path")
    if not src:
        return None
    src_path = Path(src)
    if not src_path.exists():
        return None

    if src_path.suffix.lower() in SUPPORTED_PDF_EXTS:
        page_idx = max(1, int(page_data.page_number)) - 1
        with fitz.open(str(src_path)) as doc:
            if page_idx >= len(doc):
                return None
            pix = doc[page_idx].get_pixmap(dpi=PDF_DPI, alpha=False)
            return Image.frombytes("RGB", (pix.width, pix.height), pix.samples)

    if src_path.suffix.lower() in SUPPORTED_IMAGE_EXTS:
        img = Image.open(src_path)
        img.load()
        return img

    return None


def _pil_to_b64_jpeg(img: Image.Image) -> str:
    img = ImageOps.exif_transpose(img)
    if img.mode != "RGB":
        img = img.convert("RGB")
    img.thumbnail((MAX_EDGE, MAX_EDGE), Image.Resampling.LANCZOS)
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=JPEG_Q, optimize=True)
    return base64.b64encode(buf.getvalue()).decode("ascii")


def _parse_json_obj(text: str) -> Dict[str, Any]:
    if not text:
        return {}
    s = text.strip()
    m = re.match(r"```(?:json)?\s*([\s\S]*?)\s*```", s, re.IGNORECASE)
    if m:
        s = m.group(1).strip()
    start, end = s.find("{"), s.rfind("}")
    if start != -1 and end > start:
        s = s[start:end + 1]
    try:
        return json.loads(s)
    except json.JSONDecodeError:
        if repair_json is None:
            return {}
        try:
            return json.loads(repair_json(s))
        except Exception:
            return {}


# ---- Provider clients -------------------------------------------------------
# A single NHAclient instance is reused across calls when PROVIDER=nha.
_NHA_CLIENT = None


def _get_nha_client():
    global _NHA_CLIENT
    if _NHA_CLIENT is None:
        from nha_client import NHAclient  # type: ignore
        _NHA_CLIENT = NHAclient(NHA_CLIENT_ID, NHA_CLIENT_SECRET)
    return _NHA_CLIENT


def _extract_content_from_response(resp: Any) -> str:
    """NHAclient may return either a dict or an OpenAI-style object. Handle both."""
    if resp is None:
        return ""
    if isinstance(resp, str):
        return resp
    if isinstance(resp, dict):
        choices = resp.get("choices") or []
        if choices:
            msg = choices[0].get("message") or {}
            return msg.get("content") or ""
        return resp.get("content") or ""
    # Object-style (e.g. OpenAI-compatible) — best-effort attribute access.
    try:
        return resp.choices[0].message.content  # type: ignore[attr-defined]
    except Exception:
        return str(resp)


# ---- Vision call ------------------------------------------------------------
def _call_fireworks_vision(image_b64: str, user_text: str, system_text: str) -> Dict[str, Any]:
    if not FIREWORKS_API_KEY:
        return {}
    r = requests.post(
        FIREWORKS_URL,
        headers={"Authorization": f"Bearer {FIREWORKS_API_KEY}", "Content-Type": "application/json"},
        json={
            "model": FIREWORKS_VISION_MODEL,
            "messages": [
                {"role": "system", "content": system_text},
                {"role": "user", "content": [
                    {"type": "text", "text": user_text},
                    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_b64}"}},
                ]},
            ],
            "response_format": {"type": "json_object"},
            "temperature": TEMPERATURE,
            "max_tokens": NUM_PREDICT,
        },
        timeout=300,
    )
    r.raise_for_status()
    return _parse_json_obj(r.json()["choices"][0]["message"]["content"])


def _call_nha_vision(image_b64: str, user_text: str, system_text: str) -> Dict[str, Any]:
    if not (NHA_CLIENT_ID and NHA_CLIENT_SECRET):
        return {}
    nc = _get_nha_client()
    resp = nc.completion(
        model=NHA_VISION_MODEL,
        messages=[
            {"role": "system", "content": system_text},
            {"role": "user", "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_b64}"}},
                {"type": "text", "text": user_text},
            ]},
        ],
        metadata={"problem_statement": NHA_PROBLEM_STATEMENT},
    )
    return _parse_json_obj(_extract_content_from_response(resp))


def _call_vision_model(image_b64: str, user_text: str, system_text: str = VISION_SYSTEM, retries: int = 3) -> Dict[str, Any]:
    """Provider-agnostic vision call. Retries on transient HTTP errors."""
    last_err: Optional[Exception] = None
    for attempt in range(retries):
        try:
            if PROVIDER == "fireworks":
                return _call_fireworks_vision(image_b64, user_text, system_text)
            return _call_nha_vision(image_b64, user_text, system_text)
        except requests.HTTPError as e:
            code = e.response.status_code if e.response is not None else None
            if code in (429, 500, 502, 503, 504) and attempt < retries - 1:
                time.sleep(2 ** attempt + 1)
                last_err = e
                continue
            raise
        except Exception as e:
            last_err = e
            if attempt < retries - 1:
                time.sleep(1 + attempt)
                continue
            break
    if last_err:
        print(f"  vision call failed: {last_err}")
    return {}


# ---- Text call --------------------------------------------------------------
def _call_fireworks_text(system_text: str, user_text: str) -> Dict[str, Any]:
    if not FIREWORKS_API_KEY:
        return {}
    r = requests.post(
        FIREWORKS_URL,
        headers={"Authorization": f"Bearer {FIREWORKS_API_KEY}", "Content-Type": "application/json"},
        json={
            "model": FIREWORKS_TEXT_MODEL,
            "messages": [
                {"role": "system", "content": system_text},
                {"role": "user", "content": user_text},
            ],
            "response_format": {"type": "json_object"},
            "temperature": SUMMARY_TEMP,
            "max_tokens": SUMMARY_PREDICT,
        },
        timeout=600,
    )
    r.raise_for_status()
    return _parse_json_obj(r.json()["choices"][0]["message"]["content"])


def _call_nha_text(system_text: str, user_text: str) -> Dict[str, Any]:
    if not (NHA_CLIENT_ID and NHA_CLIENT_SECRET):
        return {}
    nc = _get_nha_client()
    resp = nc.completion(
        model=NHA_TEXT_MODEL,
        messages=[
            {"role": "system", "content": system_text},
            {"role": "user",   "content": user_text},
        ],
        metadata={"problem_statement": NHA_PROBLEM_STATEMENT},
    )
    return _parse_json_obj(_extract_content_from_response(resp))


def _call_text_model(system_text: str, user_text: str, retries: int = 3) -> Dict[str, Any]:
    """Provider-agnostic text call."""
    last_err: Optional[Exception] = None
    for attempt in range(retries):
        try:
            if PROVIDER == "fireworks":
                return _call_fireworks_text(system_text, user_text)
            return _call_nha_text(system_text, user_text)
        except requests.HTTPError as e:
            code = e.response.status_code if e.response is not None else None
            if code in (429, 500, 502, 503, 504) and attempt < retries - 1:
                time.sleep(2 ** attempt + 1)
                last_err = e
                continue
            raise
        except Exception as e:
            last_err = e
            if attempt < retries - 1:
                time.sleep(1 + attempt)
                continue
            break
    if last_err:
        print(f"  text call failed: {last_err}")
    return {}


# ---- Pipeline functions -----------------------------------------------------
def initialize_page_data(page_stub: Dict[str, Any]) -> PageData:
    return PageData(
        package_code=page_stub["package_code"],
        case_id=page_stub["case_id"],
        source_file=page_stub["source_file"],
        file_kind=page_stub["file_kind"],
        page_number=page_stub.get("page_number", 1),
        page_id=page_stub["page_id"],
        image_path=page_stub.get("image_path"),
        metadata={
            "file_path": page_stub.get("file_path", ""),
            "runner_initialized": True,
        },
    )


def _vision_user_instruction(page_data: PageData) -> str:
    return (
        f"IMAGE: {page_data.source_file} page {page_data.page_number}\n\n"
        "Produce the JSON object described in the system prompt. "
        "OCR every legible line verbatim. Translate non-English to English. "
        "ONE JSON object. NO PROSE."
    )


def extract_text_for_page(page_data: PageData) -> str:
    """Run the per-page vision call. Cache the full record on metadata."""
    if page_data.file_kind in {"medical_image", "pdf_document"}:
        try:
            img = _load_pil_for_page(page_data)
        except Exception as e:
            page_data.metadata["llm_error"] = f"image load failed: {e}"
            return ""
        if img is None:
            return ""
        try:
            b64 = _pil_to_b64_jpeg(img)
        except Exception as e:
            page_data.metadata["llm_error"] = f"image preprocess failed: {e}"
            return ""

        record = _call_vision_model(b64, _vision_user_instruction(page_data))
        page_data.metadata["fireworks_record"] = record  # legacy key, kept for downstream readers
        page_data.metadata["llm_record"] = record
        ocr = record.get("ocr") or {}
        text = ocr.get("english") or ocr.get("verbatim") or ""
        return text if isinstance(text, str) else ""

    if page_data.file_kind == "written_report":
        src = page_data.metadata.get("file_path") or page_data.image_path
        if not src:
            return ""
        try:
            return Path(src).read_text(encoding="utf-8", errors="replace")
        except Exception:
            return ""

    return ""


def attach_text_to_page_data(page_data: PageData, extracted_text: str) -> PageData:
    """Decide whether the OCR text is a report or a scan caption based on image_type."""
    page_data.extracted_text = extracted_text or ""
    record = page_data.metadata.get("llm_record") or page_data.metadata.get("fireworks_record") or {}
    image_type = (record.get("image_type") or "").strip()

    is_report_like = (
        page_data.file_kind == "written_report"
        or image_type in REPORT_IMAGE_TYPES
    )
    if is_report_like:
        page_data.report_text = extracted_text or ""
        page_data.metadata["effective_kind"] = "written_report"
    else:
        page_data.metadata["effective_kind"] = "medical_image"

    return page_data


In [9]:
# =========================
# 2. IMAGE QUALITY / UNCERTAINTY PIPELINE
# =========================

QUALITY_LABEL = {
    "good":  "Good",
    "fair":  "Fair",
    "poor":  "Poor",
}


def assess_image_quality(page_data: PageData) -> Dict[str, Any]:
    """Pull rating + limitations from the cached vision record."""
    record = page_data.metadata.get("fireworks_record") or {}
    iq = record.get("image_quality") or {}
    rating_raw = (iq.get("rating") or "").strip().lower()
    rating = QUALITY_LABEL.get(rating_raw, "")
    limitations = [str(x).strip() for x in (iq.get("limitations") or []) if str(x).strip()]

    if not rating and page_data.metadata.get("effective_kind") == "written_report":
        # Reports do not need an image-quality rating; mark Good for usability.
        rating = "Good"

    quality_comment = rating
    if limitations:
        quality_comment = f"{rating} - {', '.join(limitations)}" if rating else ", ".join(limitations)
    if not quality_comment:
        quality_comment = "Not assessed / Not seen"

    return {
        "rating": rating or "Not assessed",
        "limitations": limitations,
        "quality_comment": quality_comment,
        "summary": quality_comment,
        "uncertainty_note": ", ".join(limitations) if limitations else "",
        "poor_quality": rating_raw == "poor",
    }


def write_image_quality_to_page(page_data: PageData, quality_result: Dict[str, Any]) -> PageData:
    page_data.image_quality_assessment = quality_result or {}
    return page_data


In [10]:
# =========================
# 3. IMAGE INTERPRETATION PIPELINE
# =========================

# ---- Package-specific anatomy prompts ---------------------------------------
ANATOMY_SYSTEM = {
    "MC011A": r"""You are a medical-image describer for India's AB PM-JAY claims, package MC011A
(Coronary angiogram + PTCA single stent). Describe what you SEE in the cardiac
angiogram still(s). Do NOT diagnose. Do NOT approve / reject.

Rules:
- For each named coronary segment, describe stenosis briefly using the format
  "Proximal ~ <state>, Mid ~ <state>, Distal ~ <state>" with values such as
  "No disease", "Not visualised", "<percent>% stenosis", "totally occluded",
  or "normal". For LCX, also include "OM ~ <state>".
- Identify whether a stent is visualised in the IMAGE (Yes / No / Not seen).
- Identify the apparent stage_of_care: "pre-procedure", "intra-procedure",
  "post-procedure", or "uncertain". Use cues such as visible stent, balloon,
  guidewire, contrast filling, and on-screen text.
- If the image is not a coronary angiogram, set every anatomy field to
  "Not assessed / Not seen" and stent_visualization to "Not seen".

OUTPUT - exactly ONE JSON object, no markdown:
{
  "stage_of_care": "pre-procedure | intra-procedure | post-procedure | uncertain",
  "lmca": "<string>",
  "lad":  "<string>",
  "lcx":  "<string>",
  "rca":  "<string>",
  "stent_visualization": "Yes | No | Not seen"
}""",

    "MG029A": r"""You are a medical-image describer for India's AB PM-JAY claims, package MG029A
(post-procedure chest X-ray review). Describe what you SEE on the chest X-ray.
Do NOT diagnose.

Rules:
- Use short descriptive phrases that match radiology conventions.
- Use "Not assessed / Not seen" for anything outside the X-ray.

OUTPUT - exactly ONE JSON object, no markdown:
{
  "stage_of_care": "pre-procedure | intra-procedure | post-procedure | uncertain",
  "lung_fields":      "<string>",
  "cp_angles":        "<e.g. 'Right CP angle blunted', 'Both clear', 'Bilateral blunted'>",
  "hilum":            "<e.g. 'Prominent', 'Normal'>",
  "midline_shift":    "<'Seen' | 'Not seen'>",
  "cardiac_size":     "<'Normal' | 'Enlarged' | 'Borderline'>"
}""",

    "SG039C": r"""You are a medical-image describer for India's AB PM-JAY claims, package SG039C
(USG abdomen for cholecystectomy). Describe what you SEE in the ultrasound
images. Do NOT diagnose.

Rules:
- Comment on each abdominal organ briefly. Use "Not assessed / Not seen" if an
  organ is not visualised in the available stills.
- For "gall_bladder", emphasise stones / wall / lumen findings if visible.

OUTPUT - exactly ONE JSON object, no markdown:
{
  "stage_of_care": "pre-procedure | intra-procedure | post-procedure | uncertain",
  "liver":              "<string>",
  "gall_bladder":       "<string>",
  "spleen":             "<string>",
  "kidneys":            "<string>",
  "urinary_bladder":    "<string>",
  "prostate_or_uterus": "<string>",
  "peritoneal_fluid":   "<'Present' | 'Not present' | 'Not assessed / Not seen'>"
}""",

    "SU007A": r"""You are a medical-image describer for India's AB PM-JAY claims, package SU007A
(DJ stenting / PCN under GA). Describe what you SEE on the IVP / KUB / X-ray.
Do NOT diagnose.

Rules:
- Identify side (left / right / bilateral) wherever visible.
- For pre-procedure (IVP / KUB without stent), populate the pre fields and
  leave the post field as "Not assessed / Not seen".
- For post-procedure (X-ray KUB after stenting), populate dj_stent_or_pcn_tube
  and leave the pre fields as "Not assessed / Not seen".

OUTPUT - exactly ONE JSON object, no markdown:
{
  "stage_of_care": "pre-procedure | intra-procedure | post-procedure | uncertain",
  "pcs":                "<pelvicalyceal system status>",
  "ureter":             "<ureter status>",
  "urinary_bladder":    "<string>",
  "stone_visualisation":"<e.g. 'Visualised on left ureter', 'Not seen', side>",
  "radio_opaque_shadow":"<'Yes' | 'No' | 'Not assessed / Not seen'>",
  "dj_stent_or_pcn_tube":"<e.g. 'DJ stent visualised on left side', 'Not seen'>"
}""",
}


def _anatomy_user_instruction(page_data: PageData) -> str:
    return (
        f"Package: {page_data.package_code}\n"
        f"Source: {page_data.source_file} (page {page_data.page_number})\n\n"
        "Produce the JSON object described in the system prompt. "
        "ONE JSON object. NO PROSE."
    )


def _default_anatomy_for_package(package_code: str) -> Dict[str, Any]:
    nd = "Not assessed / Not seen"
    if package_code == "MC011A":
        return {"stage_of_care": "uncertain", "lmca": nd, "lad": nd, "lcx": nd, "rca": nd, "stent_visualization": "Not seen"}
    if package_code == "MG029A":
        return {"stage_of_care": "uncertain", "lung_fields": nd, "cp_angles": nd, "hilum": nd, "midline_shift": nd, "cardiac_size": nd}
    if package_code == "SG039C":
        return {"stage_of_care": "uncertain", "liver": nd, "gall_bladder": nd, "spleen": nd, "kidneys": nd, "urinary_bladder": nd, "prostate_or_uterus": nd, "peritoneal_fluid": nd}
    if package_code == "SU007A":
        return {"stage_of_care": "uncertain", "pcs": nd, "ureter": nd, "urinary_bladder": nd, "stone_visualisation": nd, "radio_opaque_shadow": nd, "dj_stent_or_pcn_tube": nd}
    return {}


def interpret_medical_image(page_data: PageData) -> Dict[str, Any]:
    """Package-specific anatomy description. Skip for report-like pages."""
    if page_data.metadata.get("effective_kind") == "written_report":
        return {}
    if page_data.file_kind not in {"medical_image", "pdf_document"}:
        return {}

    pkg = page_data.package_code
    system_text = ANATOMY_SYSTEM.get(pkg)
    if not system_text:
        return {}

    try:
        img = _load_pil_for_page(page_data)
    except Exception:
        img = None
    if img is None:
        return {}
    try:
        b64 = _pil_to_b64_jpeg(img)
    except Exception:
        return {}

    raw = _call_vision_model(b64, _anatomy_user_instruction(page_data), system_text=system_text)
    if not isinstance(raw, dict) or not raw:
        return {}

    anatomy = _default_anatomy_for_package(pkg)
    for k in list(anatomy.keys()):
        v = raw.get(k)
        if isinstance(v, str) and v.strip():
            anatomy[k] = v.strip()

    stage = (anatomy.get("stage_of_care") or "uncertain").strip().lower()
    if stage not in {"pre-procedure", "post-procedure", "intra-procedure", "uncertain"}:
        stage = "uncertain"
    anatomy["stage_of_care"] = stage

    obs: Dict[str, Any] = {"_anatomy": anatomy, "_stage_of_care": stage}

    if pkg == "MC011A":
        if stage == "post-procedure":
            obs["post_lmca"] = anatomy["lmca"]
            obs["post_lad"]  = anatomy["lad"]
            obs["post_lcx"]  = anatomy["lcx"]
            obs["post_rca"]  = anatomy["rca"]
            obs["stent_visualization"] = anatomy["stent_visualization"]
        else:
            obs["pre_lmca"] = anatomy["lmca"]
            obs["pre_lad"]  = anatomy["lad"]
            obs["pre_lcx"]  = anatomy["lcx"]
            obs["pre_rca"]  = anatomy["rca"]
    elif pkg == "MG029A":
        obs["lung_fields"]   = anatomy["lung_fields"]
        obs["cp_angles"]     = anatomy["cp_angles"]
        obs["hilum"]         = anatomy["hilum"]
        obs["midline_shift"] = anatomy["midline_shift"]
        obs["cardiac_size"]  = anatomy["cardiac_size"]
    elif pkg == "SG039C":
        for k in ("liver", "gall_bladder", "spleen", "kidneys", "urinary_bladder", "prostate_or_uterus", "peritoneal_fluid"):
            obs[k] = anatomy[k]
    elif pkg == "SU007A":
        if stage == "post-procedure":
            obs["dj_stent_or_pcn_tube"] = anatomy["dj_stent_or_pcn_tube"]
            obs["radio_opaque_shadow"] = anatomy.get("radio_opaque_shadow", "No")
        else:
            obs["pcs"] = anatomy["pcs"]
            obs["ureter"] = anatomy["ureter"]
            obs["urinary_bladder"] = anatomy["urinary_bladder"]
            obs["stone_visualisation"] = anatomy["stone_visualisation"]
            obs["radio_opaque_shadow"] = anatomy.get("radio_opaque_shadow", "No")

    return obs


def write_image_observations_to_page(page_data: PageData, image_result: Dict[str, Any]) -> PageData:
    page_data.image_observations = image_result or {}
    return page_data


In [11]:
# =========================
# 4. WRITTEN REPORT ANALYSIS PIPELINE
# =========================

REPORT_EXTRACT_SYSTEM = r"""You are a medical-report extractor for India's AB PM-JAY claims dataset.
You receive the OCR'd text of ONE radiology report. Extract the structured
content. Do NOT diagnose. Do NOT approve / reject.

Rules:
- Use null for any field you cannot derive.
- Dates as DD/MM/YYYY only.
- Modality / view as written in the report.

OUTPUT - exactly ONE JSON object, no markdown:
{
  "modality":          "<e.g. 'Chest X-Ray (PA)' | 'USG abdomen' | 'IVP' | 'Coronary angiogram' | null>",
  "study_date":        "<DD/MM/YYYY | null>",
  "admission_date":    "<DD/MM/YYYY | null>",
  "discharge_date":    "<DD/MM/YYYY | null>",
  "observations":      ["<observation line>"],
  "impression":        ["<impression line>"],
  "advice":            ["<advice line>"],
  "patient_age_sex":   "<e.g. '55Y/F' | null>"
}"""


INTRA_CONSISTENCY_SYSTEM = r"""You assess INTRA-report consistency for India's AB PM-JAY radiology claims.
Given the structured observations, impression, and advice extracted from a
radiology report, decide whether the impression and advice are supported by
the observations IN THE SAME REPORT. Do NOT diagnose.

Allowed status values: "Consistent", "Partially supported", "Unsupported".
Note must be ONE concise reviewer-friendly sentence.

OUTPUT - exactly ONE JSON object, no markdown:
{
  "status": "Consistent | Partially supported | Unsupported",
  "note":   "<one sentence reasoning>"
}"""


def analyze_written_report(page_data: PageData) -> Dict[str, Any]:
    """Run report extraction LLM call on this page's report text."""
    text = (page_data.report_text or "").strip()
    if not text and page_data.metadata.get("effective_kind") == "written_report":
        text = (page_data.extracted_text or "").strip()
    if not text:
        return {}

    user_text = (
        f"Report source: {page_data.source_file} (page {page_data.page_number})\n"
        f"Package: {page_data.package_code}\n\n"
        "OCR'd report text:\n---\n"
        f"{text[:6000]}"
    )
    raw = _call_text_model(REPORT_EXTRACT_SYSTEM, user_text)
    if not isinstance(raw, dict):
        return {}

    obs = raw.get("observations") if isinstance(raw.get("observations"), list) else []
    imp = raw.get("impression")   if isinstance(raw.get("impression"),   list) else []
    adv = raw.get("advice")       if isinstance(raw.get("advice"),       list) else []
    return {
        "modality":        (raw.get("modality") or None),
        "study_date":      (raw.get("study_date") or None),
        "admission_date":  (raw.get("admission_date") or None),
        "discharge_date":  (raw.get("discharge_date") or None),
        "observations":    [str(x).strip() for x in obs if str(x).strip()],
        "impression":      [str(x).strip() for x in imp if str(x).strip()],
        "advice":          [str(x).strip() for x in adv if str(x).strip()],
        "patient_age_sex": raw.get("patient_age_sex"),
    }


def check_internal_report_consistency(page_data: PageData) -> Dict[str, Any]:
    obs = page_data.report_observations or {}
    if not obs.get("observations") and not obs.get("impression"):
        return {}

    user_text = (
        "Structured report extract:\n"
        f"{json.dumps({k: obs.get(k) for k in ('observations', 'impression', 'advice')}, ensure_ascii=False, indent=2)}"
    )
    raw = _call_text_model(INTRA_CONSISTENCY_SYSTEM, user_text)
    if not isinstance(raw, dict) or not raw:
        return {}

    status = (raw.get("status") or "").strip()
    if status.lower() in {"consistent", "partially supported", "unsupported"}:
        status = {"consistent": "Consistent", "partially supported": "Partially supported", "unsupported": "Unsupported"}[status.lower()]
    return {
        "status": status or "Consistent",
        "note":   (raw.get("note") or "").strip(),
    }


def write_report_analysis_to_page(
    page_data: PageData,
    report_result: Dict[str, Any],
    consistency_result: Dict[str, Any],
) -> PageData:
    if isinstance(report_result, dict):
        page_data.report_observations = report_result
    if isinstance(consistency_result, dict):
        page_data.internal_report_consistency = consistency_result
    return page_data


In [12]:
# =========================
# 5. IMAGE-TO-REPORT CORRELATION PIPELINE
# =========================

INTER_CORRELATION_SYSTEM = r"""You compare image findings with the written radiology report
for India's AB PM-JAY claims dataset. Decide whether the IMAGE findings
support the REPORT statements. Do NOT diagnose. Do NOT approve / reject.

Allowed status values: "Consistent", "Partially supported", "Unsupported".
Note must be ONE concise reviewer-friendly sentence.

OUTPUT - exactly ONE JSON object, no markdown:
{
  "status": "Consistent | Partially supported | Unsupported",
  "note":   "<one sentence reasoning citing visible features>"
}"""


def correlate_image_with_report(page_data: PageData) -> Dict[str, Any]:
    """
    Page-level correlation is left empty intentionally.
    The case-level fusion pass (in run_ps2_pipeline) populates this using
    aggregated per-page evidence so a single image page can be compared
    against the merged report content of the entire case.
    """
    return {}


def write_correlation_to_page(page_data: PageData, correlation_result: Dict[str, Any]) -> PageData:
    if isinstance(correlation_result, dict) and correlation_result:
        page_data.image_report_correlation = correlation_result
    return page_data


In [13]:
# =========================
# 6. PACKAGE / STG / TIMELINE PIPELINE
# =========================

STG_ALIGNMENT_SYSTEM = r"""You evaluate Standard Treatment Guideline (STG) alignment for an
India AB PM-JAY radiology claim. The case has package code {pkg}.

Package context (for reasoning, do NOT diagnose):
- MC011A: Coronary angiogram + PTCA single stent. Pre-procedure angiogram and
  post-procedure angiogram (showing stent / restored flow) are both expected.
- MG029A: COPD exacerbation / pneumonitis follow-up; only POST-procedure chest
  X-ray expected.
- SG039C: Cholecystectomy; only PRE-procedure USG abdomen expected.
- SU007A: DJ stenting / PCN under GA; PRE-procedure IVP and POST-procedure
  X-ray KUB expected.

INPUT: a JSON object summarising the dossier.

Allowed alignment_category values:
- "Both pre procedure and post procedure investigations align"
- "Only pre-procedure investigation aligns"
- "Only post-procedure investigtaion aligns"
- "None of the investigations align"

Allowed stg_alignment_outcome values: "Consistent", "Partially supported", "Unsupported".

OUTPUT - exactly ONE JSON object, no markdown:
{
  "alignment_category":     "<from list>",
  "alignment_reason":       "<one short sentence>",
  "stg_alignment_outcome":  "Consistent | Partially supported | Unsupported",
  "stg_reason":             "<one short sentence>"
}"""


CASE_FUSION_INTER_SYSTEM = r"""You compare image findings with the written report for an
India AB PM-JAY radiology claim. The dossier may contain pre-procedure and
post-procedure images and one or more report files. Produce TWO inter
correlation verdicts: one for pre-procedure and one for post-procedure.

If the package has no pre OR no post stage, set the absent stage's status to
"Not assessed / Not seen" and note to "No pre-procedure investigation in this package."
or similar.

Allowed status values: "Consistent", "Partially supported", "Unsupported",
"Not assessed / Not seen".

OUTPUT - exactly ONE JSON object, no markdown:
{
  "pre_status": "<status>",
  "pre_note":   "<one sentence>",
  "post_status": "<status>",
  "post_note":   "<one sentence>"
}"""


def assess_package_and_stg_alignment(page_data: PageData) -> Dict[str, Any]:
    """No-op at page level. The case-level fusion populates this once per case."""
    return {}


def _parse_dmy(value: Optional[str]) -> Optional[datetime]:
    if not value or not isinstance(value, str):
        return None
    s = value.strip()
    for fmt in ("%d/%m/%Y", "%Y-%m-%d", "%d-%m-%Y", "%d.%m.%Y"):
        try:
            return datetime.strptime(s, fmt)
        except ValueError:
            continue
    m = re.search(r"(\d{1,2})[/\-.](\d{1,2})[/\-.](\d{4})", s)
    if m:
        try:
            return datetime(int(m.group(3)), int(m.group(2)), int(m.group(1)))
        except ValueError:
            return None
    return None


def _format_dmy(dt: Optional[datetime]) -> str:
    return dt.strftime("%d/%m/%Y") if isinstance(dt, datetime) else ""


def assess_timeline_and_stage_of_care(page_data: PageData) -> Dict[str, Any]:
    """Extract dates from the cached vision record and the report observations."""
    record = page_data.metadata.get("fireworks_record") or {}
    raw_dates = record.get("dates_found") or []
    parsed: List[datetime] = []
    for s in raw_dates:
        d = _parse_dmy(s)
        if d:
            parsed.append(d)

    obs = page_data.report_observations or {}

    return {
        "page_dates": [d.strftime("%d/%m/%Y") for d in parsed],
        "report_admission_date": obs.get("admission_date"),
        "report_discharge_date": obs.get("discharge_date"),
        "report_study_date":     obs.get("study_date"),
        "stage_of_care":         (record.get("stage_of_care") or "").strip().lower(),
    }


def write_context_assessments_to_page(
    page_data: PageData,
    stg_result: Dict[str, Any],
    timeline_result: Dict[str, Any],
) -> PageData:
    if isinstance(stg_result, dict):
        page_data.package_stg_alignment = stg_result
    if isinstance(timeline_result, dict):
        page_data.timeline_stage_assessment = timeline_result
    return page_data


In [ ]:

# =========================
# EXPORT / SERIALIZATION HELPERS
# =========================

def ensure_json_serializable(data: Any) -> Any:
    """
    Convert notebook output objects into JSON-serializable forms.

    Intended responsibilities:
    - Recursively convert dataclasses and Path-like objects.
    - Make final outputs safe to dump as JSON.
    - Preserve structure without changing semantic meaning.
    """
    if hasattr(data, "__dataclass_fields__"):
        return ensure_json_serializable(asdict(data))
    if isinstance(data, Path):
        return str(data)
    if isinstance(data, dict):
        return {str(k): ensure_json_serializable(v) for k, v in data.items()}
    if isinstance(data, list):
        return [ensure_json_serializable(v) for v in data]
    if isinstance(data, tuple):
        return [ensure_json_serializable(v) for v in data]
    return data


def _pick_first_nonempty(*values: Any, default: str = "") -> str:
    for value in values:
        if value is None:
            continue
        if isinstance(value, str) and value.strip():
            return value.strip()
        if not isinstance(value, str):
            return str(value)
    return default


def _normalize_date_string(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, str):
        return value.strip()
    try:
        if pd is not None and hasattr(pd, "Timestamp") and isinstance(value, pd.Timestamp):
            return value.strftime("%d/%m/%Y")
    except Exception:
        pass
    try:
        return value.strftime("%d/%m/%Y")
    except Exception:
        return str(value)



def _build_textual_summary(case_result: CaseResult, row: Dict[str, Any]) -> str:
    summary_parts: List[str] = []
    pkg = case_result.package_code
    nd = "Not assessed / Not seen"

    K_PRE_IMG_TYPE_LF  = "Type of radiological image in pre-auth/ pre-procedure stage \n(mention absent if no image)"
    K_POST_IMG_TYPE_LF = "Type of radiological image in cclaim submission/ post-procedure stage \n(mention absent if no image)"
    K_POST_IMG_TYPE_NL = "Type of radiological image in cclaim submission/ post-procedure stage\n(mention absent if no image)"
    K_INTER_PRE        = "Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)"
    K_INTER_POST       = "Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)_Post"
    K_PKG_ALIGN        = "Claim and Package Alignment \n(Both pre procedure and post procedure investigations align/ Only pre-procedure investigation aligns/ Only post-procedure investigtaion aligns/ None of the investigations align)"
    K_STG_ALIGN_A      = "Investigation and STG Alignment (Consistent/Partially supported/Unsupported)"
    K_STG_ALIGN_B      = "Investigation and STG Alignment \n(Consistent/Partially supported/Unsupported)"

    pre_img      = row.get(K_PRE_IMG_TYPE_LF, "Absent")
    post_img_lf  = row.get(K_POST_IMG_TYPE_LF, "Absent")
    post_img_nl  = row.get(K_POST_IMG_TYPE_NL, "Absent")
    inter_pre    = row.get(K_INTER_PRE, nd)
    inter_post   = row.get(K_INTER_POST, nd)
    pkg_align    = row.get(K_PKG_ALIGN, nd)
    stg_align_a  = row.get(K_STG_ALIGN_A, nd)
    stg_align_b  = row.get(K_STG_ALIGN_B, nd)

    if pkg == "MC011A":
        summary_parts.append("Pre image: " + str(pre_img) + ".")
        summary_parts.append("Post image: " + str(post_img_lf) + ".")
        summary_parts.append("Pre report correlation: " + str(inter_pre) + ".")
        summary_parts.append("Post report correlation: " + str(inter_post) + ".")
    elif pkg == "MG029A":
        lung_fields = row.get("Lung fields", nd)
        summary_parts.append("Post image: " + str(post_img_lf) + ".")
        summary_parts.append("Lung fields: " + str(lung_fields) + ".")
        summary_parts.append("Report correlation: " + str(inter_pre) + ".")
    elif pkg == "SG039C":
        gall_bladder = row.get("Gall bladder", nd)
        summary_parts.append("Pre image: " + str(pre_img) + ".")
        summary_parts.append("Gall bladder: " + str(gall_bladder) + ".")
        summary_parts.append("Report correlation: " + str(inter_pre) + ".")
    elif pkg == "SU007A":
        stone    = row.get("Stone visualisation", nd)
        dj_stent = row.get("DJ stent/ PCN tube visualised", nd)
        summary_parts.append("Pre image: " + str(pre_img) + ".")
        summary_parts.append("Post image: " + str(post_img_nl) + ".")
        summary_parts.append("Stone visualisation: " + str(stone) + ".")
        summary_parts.append("Post device evidence: " + str(dj_stent) + ".")

    if pkg == "MC011A":
        summary_parts.append("Package alignment: " + str(pkg_align) + ".")
        summary_parts.append("STG alignment: " + str(stg_align_a) + ".")
    elif pkg == "MG029A":
        summary_parts.append("Package alignment: " + str(pkg_align) + ".")
        summary_parts.append("STG alignment: " + str(stg_align_b) + ".")
    elif pkg == "SG039C":
        summary_parts.append("Package alignment: " + str(pkg_align) + ".")
        summary_parts.append("STG alignment: " + str(stg_align_a) + ".")
    elif pkg == "SU007A":
        summary_parts.append("Package alignment: " + str(pkg_align) + ".")
        summary_parts.append("STG alignment: " + str(stg_align_b) + ".")

    return " ".join(summary_parts).strip()


def _rename_field_for_export(package_code: str, field_name: str) -> str:
    return PACKAGE_EXPORT_FIELD_RENAMES.get(package_code, {}).get(field_name, field_name)


def _export_section_value(package_code: str, section_name: str, field_name: str, row: Dict[str, Any]) -> Any:
    display_name = _rename_field_for_export(package_code, field_name)

    if section_name == "1: Image classification":
        display_name = PACKAGE_IMAGE_SECTION_FIELD_LABELS.get(package_code, {}).get(field_name, display_name)
    elif section_name == "4: Stage awareness":
        display_name = display_name.replace("\n", " ")

    value = row.get(field_name, "")
    return display_name, value


def _row_to_sectioned_json(package_code: str, row: Dict[str, Any]) -> Dict[str, Any]:
    payload: Dict[str, Any] = {}
    for section_name, fields in PACKAGE_SECTION_MAP[package_code]:
        payload[section_name] = {}
        for field_name in fields:
            display_name, value = _export_section_value(package_code, section_name, field_name, row)
            payload[section_name][display_name] = value

    for trailing_field in PACKAGE_TRAILING_FIELDS.get(package_code, []):
        payload[trailing_field] = row.get(trailing_field, "")

    return payload


def _build_auto_remarks(case_result: "CaseResult") -> str:
    """Summarise notable findings into a one-line remarks string.

    Surfaces:
      - poor image quality (with limitations)
      - image-vs-report inter-correlation issues (Unsupported / Partially supported)
      - internal report consistency issues
      - STG alignment issues
      - timeline issues (pre-investigation > 1 month old, post outside window)
    Returns "" when nothing notable.
    """
    pages = case_result.pages or []
    if not pages:
        return ""

    image_pages = [p for p in pages if p.file_kind in {"medical_image", "pdf_document"}
                   and p.metadata.get("effective_kind") != "written_report"]
    report_pages = [p for p in pages if p.metadata.get("effective_kind") == "written_report"]
    image0  = image_pages[0]  if image_pages  else pages[0]
    report0 = report_pages[0] if report_pages else pages[0]

    parts: List[str] = []

    q = image0.image_quality_assessment or {}
    if (q.get("rating") or "").lower() == "poor":
        lim = ", ".join(q.get("limitations") or []) or "low usability"
        parts.append(f"Poor image quality ({lim})")

    corr = image0.image_report_correlation or {}
    pre_status  = (corr.get("pre_status") or corr.get("status") or "").lower()
    pre_note    =  corr.get("pre_note") or corr.get("note") or ""
    post_status = (corr.get("post_status") or corr.get("status") or "").lower()
    post_note   =  corr.get("post_note") or corr.get("note") or ""
    if pre_status in {"partially supported", "unsupported"}:
        parts.append(f"Pre image/report correlation {pre_status}: {pre_note}".rstrip(": ").rstrip())
    if post_status in {"partially supported", "unsupported"}:
        parts.append(f"Post image/report correlation {post_status}: {post_note}".rstrip(": ").rstrip())

    intra = report0.internal_report_consistency or {}
    intra_status = (intra.get("status") or "").lower()
    if intra_status in {"partially supported", "unsupported"}:
        parts.append(f"Report internal consistency {intra_status}: {intra.get('note', '')}".rstrip(": ").rstrip())

    align = image0.package_stg_alignment or {}
    stg_outcome = (align.get("stg_alignment_outcome") or align.get("status") or "").lower()
    if stg_outcome in {"partially supported", "unsupported"}:
        parts.append(f"STG alignment {stg_outcome}: {align.get('stg_reason') or align.get('note', '')}".rstrip(": ").rstrip())

    timeline = (pages[0].timeline_stage_assessment or {})
    if timeline.get("pre_older_than_one_month") == "Yes":
        parts.append("Pre-procedure investigation is more than one month before admission")
    if timeline.get("post_outside_admission_discharge_window") == "Yes":
        parts.append("Post-procedure investigation date falls outside admission-discharge window")

    if case_result.reviewer_flags:
        # If we already captured the same things via the verdict checks above we
        # would double-count; only add a generic flag count line when none of the
        # specific lines above triggered.
        if not parts:
            kinds = sorted({f.get("flag_type", "issue") for f in case_result.reviewer_flags})
            parts.append(f"{len(case_result.reviewer_flags)} reviewer flag(s): {', '.join(kinds)}")

    return ". ".join(p for p in parts if p).strip()


def build_package_json_row(case_result: CaseResult) -> Dict[str, Any]:
    package_code = case_result.package_code
    row = initialize_package_row(package_code, case_result.case_id)

    pages = case_result.pages
    image_pages = [p for p in pages if p.file_kind in {"medical_image", "pdf_document"}]
    report_pages = [p for p in pages if p.file_kind == "written_report"]

    page0 = pages[0] if pages else None
    image0 = image_pages[0] if image_pages else page0
    report0 = report_pages[0] if report_pages else page0

    quality_note = _pick_first_nonempty(
        (image0.image_quality_assessment or {}).get("quality_comment") if image0 else "",
        (image0.image_quality_assessment or {}).get("summary") if image0 else "",
        (image0.image_quality_assessment or {}).get("uncertainty_note") if image0 else "",
        "Not assessed / Not seen",
    )

    admission_date = _normalize_date_string((page0.timeline_stage_assessment or {}).get("admission_date") if page0 else "")
    pre_date = _normalize_date_string((page0.timeline_stage_assessment or {}).get("pre_investigation_date") if page0 else "")
    post_date = _normalize_date_string((page0.timeline_stage_assessment or {}).get("post_investigation_date") if page0 else "")
    discharge_date = _normalize_date_string((page0.timeline_stage_assessment or {}).get("discharge_date") if page0 else "")

    if package_code == "MC011A":
        if image_pages:
            row["Type of radiological image in pre-auth/ pre-procedure stage \n(mention absent if no image)"] = PACKAGE_IMAGE_TYPE_DEFAULTS["MC011A"]["pre"]
            row["Type of radiological image in cclaim submission/ post-procedure stage \n(mention absent if no image)"] = PACKAGE_IMAGE_TYPE_DEFAULTS["MC011A"]["post"]
        obs = (image0.image_observations if image0 else {}) or {}
        corr = (image0.image_report_correlation if image0 else {}) or {}
        intra = (report0.internal_report_consistency if report0 else {}) or {}
        align = (image0.package_stg_alignment if image0 else {}) or {}

        row["LMCA"] = _pick_first_nonempty(obs.get("pre_lmca"), default=row["LMCA"])
        row["LAD"] = _pick_first_nonempty(obs.get("pre_lad"), default=row["LAD"])
        row["LCX "] = _pick_first_nonempty(obs.get("pre_lcx"), default=row["LCX "])
        row["RCA"] = _pick_first_nonempty(obs.get("pre_rca"), default=row["RCA"])
        row["LMCA_Post"] = _pick_first_nonempty(obs.get("post_lmca"), default=row["LMCA_Post"])
        row["LAD_Post"] = _pick_first_nonempty(obs.get("post_lad"), default=row["LAD_Post"])
        row["LCX_Post"] = _pick_first_nonempty(obs.get("post_lcx"), default=row["LCX_Post"])
        row["RCA_Post"] = _pick_first_nonempty(obs.get("post_rca"), default=row["RCA_Post"])
        row["Stent Visualization"] = _pick_first_nonempty(obs.get("stent_visualization"), default=row["Stent Visualization"])

        row["Intra: Within report consistency check \n(Consistent/Partially supported/Unsupported)"] = _pick_first_nonempty(intra.get("pre_status"), intra.get("status"), default=row["Intra: Within report consistency check \n(Consistent/Partially supported/Unsupported)"])
        row["Reasoning_Pre_Intra"] = _pick_first_nonempty(intra.get("pre_note"), intra.get("note"), default=row["Reasoning_Pre_Intra"])
        row["Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)"] = _pick_first_nonempty(corr.get("pre_status"), corr.get("status"), default=row["Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)"])
        row["Reasoning_Pre_Inter"] = _pick_first_nonempty(corr.get("pre_note"), corr.get("note"), default=row["Reasoning_Pre_Inter"])
        row["Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)_Post"] = _pick_first_nonempty(corr.get("post_status"), corr.get("status"), default=row["Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)_Post"])
        row["Reasoning_Post_Inter"] = _pick_first_nonempty(corr.get("post_note"), corr.get("note"), default=row["Reasoning_Post_Inter"])

        row["Claim and Package Alignment \n(Both pre procedure and post procedure investigations align/ Only pre-procedure investigation aligns/ Only post-procedure investigtaion aligns/ None of the investigations align)"] = _pick_first_nonempty(align.get("alignment_category"), default=row["Claim and Package Alignment \n(Both pre procedure and post procedure investigations align/ Only pre-procedure investigation aligns/ Only post-procedure investigtaion aligns/ None of the investigations align)"])
        row["Reasoning_Alignment"] = _pick_first_nonempty(align.get("alignment_reason"), align.get("note"), default=row["Reasoning_Alignment"])
        row["Investigation and STG Alignment (Consistent/Partially supported/Unsupported)"] = _pick_first_nonempty(align.get("stg_alignment_outcome"), align.get("status"), default=row["Investigation and STG Alignment (Consistent/Partially supported/Unsupported)"])
        row["Reason for deviation from STG (to be mapped to PPD and CPD section)"] = _pick_first_nonempty(align.get("stg_reason"), align.get("note"), default=row["Reason for deviation from STG (to be mapped to PPD and CPD section)"])

        row["Admission Date\n(DD/MM/YYYY)"] = admission_date
        row["Pre-procedure investigation date (From written report, or else the radiological image)\n(DD/MM/YYYY)"] = pre_date
        row["Post Procedure Investigation Date (From written report, or else the radiological image)\n(DD/MM/YYYY)"] = post_date
        row["Discharge Date\n(DD/MM/YYYY)"] = discharge_date
        row["Flag for one month older documents for pre-procedure investigations \n(Yes/ No)"] = _pick_first_nonempty((page0.timeline_stage_assessment or {}).get("pre_older_than_one_month") if page0 else "", default=row["Flag for one month older documents for pre-procedure investigations \n(Yes/ No)"])
        row["Flag before admssion date or after discharge date documents for post-procedure investigations \n(Yes - outside the window/\nNo - within the window)"] = _pick_first_nonempty((page0.timeline_stage_assessment or {}).get("post_outside_admission_discharge_window") if page0 else "", default=row["Flag before admssion date or after discharge date documents for post-procedure investigations \n(Yes - outside the window/\nNo - within the window)"])
        row["Comment on image quality/ usability"] = quality_note
        row["Remarks"] = _pick_first_nonempty(
            (image0.reviewer_notes or {}).get("remarks") if image0 else "",
            _build_auto_remarks(case_result),
            default=row["Remarks"],
        )

    elif package_code == "MG029A":
        if image_pages:
            row["Type of radiological image in cclaim submission/ post-procedure stage \n(mention absent if no image)"] = PACKAGE_IMAGE_TYPE_DEFAULTS["MG029A"]["post"]
        obs = (image0.image_observations if image0 else {}) or {}
        corr = (image0.image_report_correlation if image0 else {}) or {}
        intra = (report0.internal_report_consistency if report0 else {}) or {}
        align = (image0.package_stg_alignment if image0 else {}) or {}

        row["Lung fields"] = _pick_first_nonempty(obs.get("lung_fields"), default=row["Lung fields"])
        row["Left and right CP angles"] = _pick_first_nonempty(obs.get("cp_angles"), default=row["Left and right CP angles"])
        row["Left and right hilum"] = _pick_first_nonempty(obs.get("hilum"), default=row["Left and right hilum"])
        row["Midline shift"] = _pick_first_nonempty(obs.get("midline_shift"), default=row["Midline shift"])
        row["Cardiac size"] = _pick_first_nonempty(obs.get("cardiac_size"), default=row["Cardiac size"])

        row["Intra Report Consistency Check (within report)\n(Consistent/Partially supported/Unsupported)"] = _pick_first_nonempty(intra.get("status"), default=row["Intra Report Consistency Check (within report)\n(Consistent/Partially supported/Unsupported)"])
        row["Reasoning_Intra"] = _pick_first_nonempty(intra.get("note"), default=row["Reasoning_Intra"])
        row["Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)"] = _pick_first_nonempty(corr.get("status"), default=row["Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)"])
        row["Reasoning_Inter"] = _pick_first_nonempty(corr.get("note"), default=row["Reasoning_Inter"])

        row["Claim and Package Alignment \n(Both pre procedure and post procedure investigations align/ Only pre-procedure investigation aligns/ Only post-procedure investigtaion aligns/ None of the investigations align)"] = _pick_first_nonempty(align.get("alignment_category"), default=row["Claim and Package Alignment \n(Both pre procedure and post procedure investigations align/ Only pre-procedure investigation aligns/ Only post-procedure investigtaion aligns/ None of the investigations align)"])
        row["Reason for Non alignment"] = _pick_first_nonempty(align.get("alignment_reason"), align.get("note"), default=row["Reason for Non alignment"])
        row["Investigation and STG Alignment \n(Consistent/Partially supported/Unsupported)"] = _pick_first_nonempty(align.get("stg_alignment_outcome"), align.get("status"), default=row["Investigation and STG Alignment \n(Consistent/Partially supported/Unsupported)"])
        row["Reason for deviation from STG \n(to be mapped to PPD and CPD section)"] = _pick_first_nonempty(align.get("stg_reason"), align.get("note"), default=row["Reason for deviation from STG \n(to be mapped to PPD and CPD section)"])

        row["Admission Date\n(DD/MM/YYYY)"] = admission_date
        row["Post Procedure Investigation Date (From written report, or else the radiological image)\n(DD/MM/YYYY)"] = post_date
        row["Discharge Date\n(DD/MM/YYYY)"] = discharge_date
        row["Flag before admssion date or after discharge date documents for post-procedure investigations \n(Yes - outside the window/\nNo - within the window)"] = _pick_first_nonempty((page0.timeline_stage_assessment or {}).get("post_outside_admission_discharge_window") if page0 else "", default=row["Flag before admssion date or after discharge date documents for post-procedure investigations \n(Yes - outside the window/\nNo - within the window)"])
        row["Comment on image quality/ usability"] = quality_note

    elif package_code == "SG039C":
        if image_pages:
            row["Type of radiological image in pre-auth/ pre-procedure stage \n(mention absent if no image)"] = PACKAGE_IMAGE_TYPE_DEFAULTS["SG039C"]["pre"]
        obs = (image0.image_observations if image0 else {}) or {}
        corr = (image0.image_report_correlation if image0 else {}) or {}
        intra = (report0.internal_report_consistency if report0 else {}) or {}
        align = (image0.package_stg_alignment if image0 else {}) or {}

        row["Liver"] = _pick_first_nonempty(obs.get("liver"), default=row["Liver"])
        row["Gall bladder"] = _pick_first_nonempty(obs.get("gall_bladder"), default=row["Gall bladder"])
        row["Spleen"] = _pick_first_nonempty(obs.get("spleen"), default=row["Spleen"])
        row["Kidneys"] = _pick_first_nonempty(obs.get("kidneys"), default=row["Kidneys"])
        row["Urinary bladder"] = _pick_first_nonempty(obs.get("urinary_bladder"), default=row["Urinary bladder"])
        row["Prostate/ Uterus"] = _pick_first_nonempty(obs.get("prostate_or_uterus"), default=row["Prostate/ Uterus"])
        row["Peritoneal fluid"] = _pick_first_nonempty(obs.get("peritoneal_fluid"), default=row["Peritoneal fluid"])

        row["Intra:Within report consistency check (Consistent/Partially supported/Unsupportred"] = _pick_first_nonempty(intra.get("status"), default=row["Intra:Within report consistency check (Consistent/Partially supported/Unsupportred"])
        row["Reasoning_Intra"] = _pick_first_nonempty(intra.get("note"), default=row["Reasoning_Intra"])
        row["Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)"] = _pick_first_nonempty(corr.get("status"), default=row["Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)"])
        row["Reason for Partially supported/Unsupported"] = _pick_first_nonempty(corr.get("note"), default=row["Reason for Partially supported/Unsupported"])

        row["Claim and Package Alignment \n(Both pre procedure and post procedure investigations align/ Only pre-procedure investigation aligns/ Only post-procedure investigtaion aligns/ None of the investigations align)"] = _pick_first_nonempty(align.get("alignment_category"), default=row["Claim and Package Alignment \n(Both pre procedure and post procedure investigations align/ Only pre-procedure investigation aligns/ Only post-procedure investigtaion aligns/ None of the investigations align)"])
        row["Reasoning_Alignment"] = _pick_first_nonempty(align.get("alignment_reason"), align.get("note"), default=row["Reasoning_Alignment"])
        row["Investigation and STG Alignment (Consistent/Partially supported/Unsupported)"] = _pick_first_nonempty(align.get("stg_alignment_outcome"), align.get("status"), default=row["Investigation and STG Alignment (Consistent/Partially supported/Unsupported)"])
        row["Reason for deviation from STG (to be mapped to PPD and CPD section)"] = _pick_first_nonempty(align.get("stg_reason"), align.get("note"), default=row["Reason for deviation from STG (to be mapped to PPD and CPD section)"])

        row["Admission Date\n(DD/MM/YYYY)"] = admission_date
        row["Pre-procedure investigation date\n(DD/MM/YYYY)"] = pre_date
        row["Discharge Date\n(DD/MM/YYYY)"] = discharge_date
        row["Flag for one month older documents for pre-procedure investigations "] = _pick_first_nonempty((page0.timeline_stage_assessment or {}).get("pre_older_than_one_month") if page0 else "", default=row["Flag for one month older documents for pre-procedure investigations "])
        row["Comment on image quality/ usability"] = quality_note

    elif package_code == "SU007A":
        if image_pages:
            row["Type of radiological image in pre-auth/ pre-procedure stage \n(mention absent if no image)"] = PACKAGE_IMAGE_TYPE_DEFAULTS["SU007A"]["pre"]
            row["Type of radiological image in cclaim submission/ post-procedure stage\n(mention absent if no image)"] = PACKAGE_IMAGE_TYPE_DEFAULTS["SU007A"]["post"]
        obs = (image0.image_observations if image0 else {}) or {}
        corr = (image0.image_report_correlation if image0 else {}) or {}
        intra = (report0.internal_report_consistency if report0 else {}) or {}
        align = (image0.package_stg_alignment if image0 else {}) or {}

        row["Pelvicalyceal system"] = _pick_first_nonempty(obs.get("pcs"), default=row["Pelvicalyceal system"])
        row["Ureter"] = _pick_first_nonempty(obs.get("ureter"), default=row["Ureter"])
        row["Urinary bladder"] = _pick_first_nonempty(obs.get("urinary_bladder"), default=row["Urinary bladder"])
        row["Stone visualisation"] = _pick_first_nonempty(obs.get("stone_visualisation"), default=row["Stone visualisation"])
        row["Any radio opaque shadow or calcification in abdomen region"] = _pick_first_nonempty(obs.get("radio_opaque_shadow"), default=row["Any radio opaque shadow or calcification in abdomen region"])
        row["DJ stent/ PCN tube visualised"] = _pick_first_nonempty(obs.get("dj_stent_or_pcn_tube"), default=row["DJ stent/ PCN tube visualised"])

        row["Intra: Within report consistency check\n (Consistent/Partially supported/Unsupported)"] = _pick_first_nonempty(intra.get("pre_status"), intra.get("status"), default=row["Intra: Within report consistency check\n (Consistent/Partially supported/Unsupported)"])
        row["Reasoning_Pre_Intra"] = _pick_first_nonempty(intra.get("pre_note"), intra.get("note"), default=row["Reasoning_Pre_Intra"])
        row["Inter: Image vs Written Report Correlation \n (Consistent/Partially supported/Unsupported)"] = _pick_first_nonempty(corr.get("pre_status"), corr.get("status"), default=row["Inter: Image vs Written Report Correlation \n (Consistent/Partially supported/Unsupported)"])
        row["Reasoning_Pre_Inter"] = _pick_first_nonempty(corr.get("pre_note"), corr.get("note"), default=row["Reasoning_Pre_Inter"])
        row["Intra: Within report consistency check\n (Consistent/Partially supported/Unsupported)_Post"] = _pick_first_nonempty(intra.get("post_status"), intra.get("status"), default=row["Intra: Within report consistency check\n (Consistent/Partially supported/Unsupported)_Post"])
        row["Reasoning_Post_Intra"] = _pick_first_nonempty(intra.get("post_note"), intra.get("note"), default=row["Reasoning_Post_Intra"])
        row["Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)_Post"] = _pick_first_nonempty(corr.get("post_status"), corr.get("status"), default=row["Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)_Post"])
        row["Reasoning_Post_Inter"] = _pick_first_nonempty(corr.get("post_note"), corr.get("note"), default=row["Reasoning_Post_Inter"])

        row["Claim and Package Alignment \n(Both pre procedure and post procedure investigations align/ Only pre-procedure investigation aligns/ Only post-procedure investigtaion aligns/ None of the investigations align)"] = _pick_first_nonempty(align.get("alignment_category"), default=row["Claim and Package Alignment \n(Both pre procedure and post procedure investigations align/ Only pre-procedure investigation aligns/ Only post-procedure investigtaion aligns/ None of the investigations align)"])
        row["Reasoning_Alignment"] = _pick_first_nonempty(align.get("alignment_reason"), align.get("note"), default=row["Reasoning_Alignment"])
        row["Investigation and STG Alignment \n(Consistent/Partially supported/Unsupported)"] = _pick_first_nonempty(align.get("stg_alignment_outcome"), align.get("status"), default=row["Investigation and STG Alignment \n(Consistent/Partially supported/Unsupported)"])
        row["Reason for deviation from STG (to be mapped to PPD and CPD section)"] = _pick_first_nonempty(align.get("stg_reason"), align.get("note"), default=row["Reason for deviation from STG (to be mapped to PPD and CPD section)"])

        row["Admission Date\n(DD/MM/YYYY)"] = admission_date
        row["Pre-procedure investigation date\n(DD/MM/YYYY)"] = pre_date
        row["Post Procedure Investigation Date\n(DD/MM/YYYY)"] = post_date
        row["Discharge Date\n(DD/MM/YYYY)"] = discharge_date
        row["Flag for one month older documents for pre-procedure investigations "] = _pick_first_nonempty((page0.timeline_stage_assessment or {}).get("pre_older_than_one_month") if page0 else "", default=row["Flag for one month older documents for pre-procedure investigations "])
        row["Flag before admssion date or after discharge date documents for post-procedure investigations \n(Yes - outside the window/\nNo - within the window)"] = _pick_first_nonempty((page0.timeline_stage_assessment or {}).get("post_outside_admission_discharge_window") if page0 else "", default=row["Flag before admssion date or after discharge date documents for post-procedure investigations \n(Yes - outside the window/\nNo - within the window)"])
        row["Comment on image quality/ usability"] = quality_note
        row["Remarks"] = _pick_first_nonempty(
            (image0.reviewer_notes or {}).get("remarks") if image0 else "",
            _build_auto_remarks(case_result),
            default=row["Remarks"],
        )

    row["Summary PDF"] = _build_textual_summary(case_result, row)
    return _row_to_sectioned_json(package_code, row)



def _extract_summary_pdf(structured_output: Dict[str, Any]) -> str:
    if not isinstance(structured_output, dict):
        return ""
    summary = structured_output.get("Summary PDF", "")
    return summary if isinstance(summary, str) else str(summary)


def write_run_outputs(run_outputs: RunOutputs, output_root: Path) -> None:
    """
    Write final notebook outputs to disk.

    Intended responsibilities:
    - Export package-specific structured rows into JSON files.
    - Keep one consolidated JSON containing all package outputs.
    - Preserve case-level summaries and reviewer flags for inspection.
    """
    output_root.mkdir(parents=True, exist_ok=True)

    package_dir = output_root / "package_json_outputs"
    package_dir.mkdir(parents=True, exist_ok=True)

    package_written_files: Dict[str, str] = {}
    consolidated_payload = {
        "package_outputs": {},
        "final_reviewer_flags": ensure_json_serializable(run_outputs.final_reviewer_flags),
        "final_case_summaries": ensure_json_serializable(run_outputs.final_case_summaries),
    }

    for package_code, rows in run_outputs.package_json_rows.items():
        file_path = package_dir / f"{package_code}_outputs.json"
        with open(file_path, "w", encoding="utf-8") as f:
            json.dump(ensure_json_serializable(rows), f, indent=2, ensure_ascii=False)
        package_written_files[package_code] = str(file_path)
        consolidated_payload["package_outputs"][package_code] = ensure_json_serializable(rows)

    consolidated_path = output_root / "ps2_all_package_outputs.json"
    with open(consolidated_path, "w", encoding="utf-8") as f:
        json.dump(consolidated_payload, f, indent=2, ensure_ascii=False)

    run_outputs.written_files = {
        "package_json_files": package_written_files,
        "consolidated_json": str(consolidated_path),
    }


In [15]:

# =========================
# MAIN RUNNER
# =========================

import sys as _sys
import time as _time

EXPECTED_PRE_PACKAGES  = {"MC011A", "SG039C", "SU007A"}
EXPECTED_POST_PACKAGES = {"MC011A", "MG029A", "SU007A"}


def _log(msg: str = "", end: str = "\n") -> None:
    _sys.stdout.write(msg + end)
    _sys.stdout.flush()


def _aggregate_case_anatomy(case_result: "CaseResult") -> Dict[str, Any]:
    placeholder = "Not assessed / Not seen"
    fused: Dict[str, Any] = {}
    image_pages = [p for p in case_result.pages if p.file_kind in {"medical_image", "pdf_document"}
                   and p.metadata.get("effective_kind") != "written_report"]
    for p in image_pages:
        for k, v in (p.image_observations or {}).items():
            if k.startswith("_") or not isinstance(v, str):
                continue
            existing = fused.get(k, "")
            if (not existing or existing == placeholder) and v and v != placeholder:
                fused[k] = v
    return fused


def _aggregate_case_dates(case_result: "CaseResult") -> Dict[str, Any]:
    pre_dates: List[datetime] = []
    post_dates: List[datetime] = []
    admission: Optional[datetime] = None
    discharge: Optional[datetime] = None
    for p in case_result.pages:
        tl = p.timeline_stage_assessment or {}
        stage = (tl.get("stage_of_care") or "").lower()
        for s in tl.get("page_dates") or []:
            d = _parse_dmy(s)
            if not d: continue
            if "post" in stage: post_dates.append(d)
            elif "pre" in stage: pre_dates.append(d)
            elif p.metadata.get("effective_kind") != "written_report":
                pre_dates.append(d)
        adm = _parse_dmy(tl.get("report_admission_date"))
        dis = _parse_dmy(tl.get("report_discharge_date"))
        if adm and (admission is None or adm < admission): admission = adm
        if dis and (discharge is None or dis > discharge): discharge = dis
    pre_date  = min(pre_dates)  if pre_dates  else None
    post_date = max(post_dates) if post_dates else None
    pre_older = "Yes" if (admission and pre_date and (admission - pre_date) > timedelta(days=31)) else "No"
    post_outside = "No"
    if post_date and admission and post_date < admission: post_outside = "Yes"
    if post_date and discharge and post_date > discharge: post_outside = "Yes"
    return {
        "admission_date": _format_dmy(admission),
        "pre_investigation_date": _format_dmy(pre_date),
        "post_investigation_date": _format_dmy(post_date),
        "discharge_date": _format_dmy(discharge),
        "pre_older_than_one_month": pre_older,
        "post_outside_admission_discharge_window": post_outside,
    }


def _fuse_case_correlation(case_result: "CaseResult") -> Dict[str, Any]:
    image_pages = [p for p in case_result.pages if p.file_kind in {"medical_image", "pdf_document"}
                   and p.metadata.get("effective_kind") != "written_report"]
    report_pages = [p for p in case_result.pages if p.metadata.get("effective_kind") == "written_report"]
    image_summary = []
    for p in image_pages:
        rec = p.metadata.get("llm_record") or p.metadata.get("fireworks_record") or {}
        image_summary.append({
            "source": p.source_file, "page": p.page_number,
            "image_type": rec.get("image_type"), "stage_of_care": rec.get("stage_of_care"),
            "summary": rec.get("summary"), "key_observations": rec.get("key_observations") or [],
            "anatomy": (p.image_observations or {}).get("_anatomy") or {k: v for k, v in (p.image_observations or {}).items() if not k.startswith("_")},
        })
    report_summary = []
    for p in report_pages:
        ro = p.report_observations or {}
        report_summary.append({"source": p.source_file, "page": p.page_number, "modality": ro.get("modality"),
                               "study_date": ro.get("study_date"), "observations": ro.get("observations") or [],
                               "impression": ro.get("impression") or [], "advice": ro.get("advice") or []})
    if not image_summary and not report_summary:
        return {}
    user_text = (
        f"Package: {case_result.package_code}\nCase: {case_result.case_id}\n\n"
        "Image evidence:\n" + json.dumps(image_summary, ensure_ascii=False, indent=2)
        + "\n\nReport evidence:\n" + json.dumps(report_summary, ensure_ascii=False, indent=2)
        + "\n\nProduce the JSON described in the system prompt. ONE JSON object."
    )
    raw = _call_text_model(CASE_FUSION_INTER_SYSTEM, user_text)
    return raw if isinstance(raw, dict) else {}


def _fuse_case_stg_alignment(case_result: "CaseResult") -> Dict[str, Any]:
    pkg = case_result.package_code
    image_pages = [p for p in case_result.pages if p.file_kind in {"medical_image", "pdf_document"}
                   and p.metadata.get("effective_kind") != "written_report"]
    report_pages = [p for p in case_result.pages if p.metadata.get("effective_kind") == "written_report"]
    has_pre  = any((p.metadata.get("llm_record") or p.metadata.get("fireworks_record") or {}).get("stage_of_care") == "pre-procedure" for p in image_pages)
    has_post = any((p.metadata.get("llm_record") or p.metadata.get("fireworks_record") or {}).get("stage_of_care") == "post-procedure" for p in image_pages)
    has_any_image = bool(image_pages); has_report = bool(report_pages)
    if not has_pre and not has_post and has_any_image:
        if pkg in {"MG029A"}: has_post = True
        if pkg in {"SG039C"}: has_pre = True
    image_anatomy_sample = []
    for p in image_pages[:3]:
        rec = p.metadata.get("llm_record") or p.metadata.get("fireworks_record") or {}
        image_anatomy_sample.append({
            "source": p.source_file, "image_type": rec.get("image_type"),
            "stage_of_care": rec.get("stage_of_care"), "summary": rec.get("summary"),
            "anatomy": (p.image_observations or {}).get("_anatomy") or {k: v for k, v in (p.image_observations or {}).items() if not k.startswith("_")},
        })
    report_sample = [{"source": p.source_file, "modality": (p.report_observations or {}).get("modality"),
                      "impression": (p.report_observations or {}).get("impression") or []} for p in report_pages[:3]]
    user_text = (
        f"Package: {pkg}\nCase: {case_result.case_id}\n"
        f"Has pre-procedure image: {has_pre}\nHas post-procedure image: {has_post}\nHas report: {has_report}\n\n"
        "Image evidence:\n" + json.dumps(image_anatomy_sample, ensure_ascii=False, indent=2)
        + "\n\nReport evidence:\n" + json.dumps(report_sample, ensure_ascii=False, indent=2)
        + "\n\nProduce the JSON described in the system prompt. ONE JSON object."
    )
    raw = _call_text_model(STG_ALIGNMENT_SYSTEM.replace("{pkg}", pkg), user_text)
    return raw if isinstance(raw, dict) else {}


def _apply_case_fusion(case_result: "CaseResult") -> None:
    if not case_result.pages: return
    image_pages = [p for p in case_result.pages if p.file_kind in {"medical_image", "pdf_document"}
                   and p.metadata.get("effective_kind") != "written_report"]
    report_pages = [p for p in case_result.pages if p.metadata.get("effective_kind") == "written_report"]
    page0 = case_result.pages[0]
    image0 = image_pages[0] if image_pages else page0
    report0 = report_pages[0] if report_pages else page0

    fused_anatomy = _aggregate_case_anatomy(case_result)
    merged_obs = dict(image0.image_observations or {})
    for k, v in fused_anatomy.items():
        if not merged_obs.get(k): merged_obs[k] = v
    image0.image_observations = merged_obs

    correlation = _fuse_case_correlation(case_result)
    if correlation: image0.image_report_correlation = correlation
    alignment = _fuse_case_stg_alignment(case_result)
    if alignment: image0.package_stg_alignment = alignment

    for rp in report_pages:
        irc = rp.internal_report_consistency or {}
        if irc.get("status"):
            report0.internal_report_consistency = irc
            break

    dates = _aggregate_case_dates(case_result)
    timeline = dict(page0.timeline_stage_assessment or {})
    timeline.update(dates)
    page0.timeline_stage_assessment = timeline

    rating_rank = {"Good": 1, "Fair": 2, "Poor": 3, "Not assessed": 0, "": 0}
    worst_rating = None
    chosen_quality: Dict[str, Any] = {}
    for p in image_pages or [page0]:
        q = p.image_quality_assessment or {}
        rating = q.get("rating") or ""
        if rating_rank.get(rating, 0) >= rating_rank.get(worst_rating or "", 0):
            worst_rating = rating
            chosen_quality = q
    if chosen_quality: image0.image_quality_assessment = chosen_quality


def _write_package_partial(output_root: Path, package_code: str, rows: List[Dict[str, Any]]) -> Path:
    """Atomic-ish flush: write to a temp file then replace, so a crash mid-write
    cannot corrupt the previous good copy."""
    pkg_dir = output_root / "package_json_outputs"
    pkg_dir.mkdir(parents=True, exist_ok=True)
    target = pkg_dir / f"{package_code}_outputs.json"
    tmp    = pkg_dir / f"{package_code}_outputs.json.tmp"
    payload = ensure_json_serializable(rows)
    tmp.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    tmp.replace(target)
    return target


def run_ps2_pipeline(input_root: Path) -> RunOutputs:
    """Discover cases, run page-level pipeline, then case-level fusion.

    Writes each package's JSON file to disk INCREMENTALLY as cases finish,
    so a mid-run crash still leaves valid partial output.
    """

    def _safe_call(fn, *args, **kwargs):
        try:
            return fn(*args, **kwargs)
        except Exception as exc:
            return {"_runner_error": f"{fn.__name__} failed: {exc}"}

    def _fallback_initialize_page_data(page_stub: Dict[str, Any]) -> PageData:
        return PageData(
            package_code=page_stub["package_code"], case_id=page_stub["case_id"],
            source_file=page_stub["source_file"], file_kind=page_stub["file_kind"],
            page_number=page_stub.get("page_number", 1), page_id=page_stub["page_id"],
            image_path=page_stub.get("image_path"),
            metadata={"file_path": page_stub.get("file_path", ""), "runner_initialized": True},
        )

    def _default_reviewer_flags(page_data: PageData) -> List[Dict[str, Any]]:
        flags = []
        if (page_data.image_quality_assessment or {}).get("poor_quality"):
            flags.append({"package_code": page_data.package_code, "case_id": page_data.case_id,
                          "page_id": page_data.page_id, "flag_type": "image_quality",
                          "message": (page_data.image_quality_assessment or {}).get("uncertainty_note", "Image quality issue flagged.")})
        for src_attr, flag_type in [("image_report_correlation", "image_report_correlation"),
                                     ("internal_report_consistency", "internal_report_consistency")]:
            d = getattr(page_data, src_attr) or {}
            if d.get("status") in {"Unsupported", "unsupported"}:
                flags.append({"package_code": page_data.package_code, "case_id": page_data.case_id,
                              "page_id": page_data.page_id, "flag_type": flag_type,
                              "message": d.get("note", "Issue flagged.")})
        return flags

    def _default_case_level_summary(case_result: CaseResult) -> Dict[str, Any]:
        return {"package_code": case_result.package_code, "case_id": case_result.case_id,
                "num_pages": len(case_result.pages), "num_flags": len(case_result.reviewer_flags),
                "overall_note": case_result.textual_summary or ""}

    pipeline_t0 = _time.time()
    _log(f"\n{'=' * 60}")
    _log(f"PS2 PIPELINE START  (provider={PROVIDER}, input={input_root})")
    _log(f"output: {OUTPUT_ROOT.resolve()}")
    _log(f"{'=' * 60}")

    discovered_files = _safe_call(discover_case_files, input_root)
    if not isinstance(discovered_files, list) or len(discovered_files) == 0:
        discovered_files = []

    from collections import defaultdict
    files_by_case: Dict[Tuple[str, str], List["DiscoveredFile"]] = defaultdict(list)
    for d in discovered_files:
        files_by_case[(d.package_code, d.case_id)].append(d)

    total_cases = len(files_by_case)
    _log(f"discovered {len(discovered_files)} files across {total_cases} cases")
    _log(f"{'=' * 60}\n")

    # Set up the output container so we can flush partials per package.
    run_outputs = RunOutputs(
        all_case_results=[],
        package_json_rows={p: [] for p in PACKAGE_OUTPUT_FIELDS.keys()},
    )

    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

    for case_idx, (case_key, case_files) in enumerate(sorted(files_by_case.items()), start=1):
        pkg, case_id = case_key
        case_t0 = _time.time()
        case_result = CaseResult(package_code=pkg, case_id=case_id)

        all_stubs: List[Tuple["DiscoveredFile", Dict[str, Any]]] = []
        for d in case_files:
            stubs = _safe_call(split_file_into_pages, d)
            if not isinstance(stubs, list) or len(stubs) == 0:
                stubs = [{
                    "package_code": d.package_code, "case_id": d.case_id,
                    "source_file": d.file_name, "file_kind": d.file_kind,
                    "page_number": 1, "page_id": f"{d.package_code}::{d.case_id}::{d.file_name}::1",
                    "image_path": d.file_path if d.file_kind in {"medical_image", "pdf_document"} else None,
                    "file_path": d.file_path,
                }]
            for s in stubs:
                all_stubs.append((d, s))
        total_pages = len(all_stubs)

        _log(f"[{case_idx}/{total_cases}] {pkg}/{case_id}  ({len(case_files)} files, {total_pages} pages)")

        for page_idx, (d, page_stub) in enumerate(all_stubs, start=1):
            page_stub.setdefault("package_code", d.package_code)
            page_stub.setdefault("case_id", d.case_id)
            page_stub.setdefault("source_file", d.file_name)
            page_stub.setdefault("file_kind", d.file_kind)
            page_stub.setdefault("page_number", 1)
            page_stub.setdefault("page_id", f"{d.package_code}::{d.case_id}::{d.file_name}::1")
            page_stub.setdefault("image_path", d.file_path if d.file_kind in {"medical_image", "pdf_document"} else None)
            page_stub.setdefault("file_path", d.file_path)

            page_t0 = _time.time()
            page_data = _safe_call(initialize_page_data, page_stub)
            if not isinstance(page_data, PageData):
                page_data = _fallback_initialize_page_data(page_stub)

            extracted_text = _safe_call(extract_text_for_page, page_data)
            if isinstance(extracted_text, str):
                w = _safe_call(attach_text_to_page_data, page_data, extracted_text)
                if isinstance(w, PageData): page_data = w
                else: page_data.extracted_text = extracted_text

            quality_result = _safe_call(assess_image_quality, page_data)
            if isinstance(quality_result, dict):
                w = _safe_call(write_image_quality_to_page, page_data, quality_result)
                page_data = w if isinstance(w, PageData) else page_data
                if not isinstance(w, PageData): page_data.image_quality_assessment = quality_result

            image_result = _safe_call(interpret_medical_image, page_data)
            if isinstance(image_result, dict):
                w = _safe_call(write_image_observations_to_page, page_data, image_result)
                page_data = w if isinstance(w, PageData) else page_data
                if not isinstance(w, PageData): page_data.image_observations = image_result

            report_result = _safe_call(analyze_written_report, page_data)
            consistency_result = _safe_call(check_internal_report_consistency, page_data)
            if isinstance(report_result, dict) or isinstance(consistency_result, dict):
                w = _safe_call(write_report_analysis_to_page, page_data,
                               report_result if isinstance(report_result, dict) else {},
                               consistency_result if isinstance(consistency_result, dict) else {})
                page_data = w if isinstance(w, PageData) else page_data
                if not isinstance(w, PageData):
                    if isinstance(report_result, dict): page_data.report_observations = report_result
                    if isinstance(consistency_result, dict): page_data.internal_report_consistency = consistency_result

            correlation_result = _safe_call(correlate_image_with_report, page_data)
            if isinstance(correlation_result, dict):
                w = _safe_call(write_correlation_to_page, page_data, correlation_result)
                page_data = w if isinstance(w, PageData) else page_data
                if not isinstance(w, PageData): page_data.image_report_correlation = correlation_result

            stg_result = _safe_call(assess_package_and_stg_alignment, page_data)
            timeline_result = _safe_call(assess_timeline_and_stage_of_care, page_data)
            if isinstance(stg_result, dict) or isinstance(timeline_result, dict):
                w = _safe_call(write_context_assessments_to_page, page_data,
                               stg_result if isinstance(stg_result, dict) else {},
                               timeline_result if isinstance(timeline_result, dict) else {})
                page_data = w if isinstance(w, PageData) else page_data
                if not isinstance(w, PageData):
                    if isinstance(stg_result, dict): page_data.package_stg_alignment = stg_result
                    if isinstance(timeline_result, dict): page_data.timeline_stage_assessment = timeline_result

            page_dt_ms = int((_time.time() - page_t0) * 1000)
            rec = page_data.metadata.get("llm_record") or page_data.metadata.get("fireworks_record") or {}
            kind = page_data.metadata.get("effective_kind") or page_data.file_kind
            label = rec.get("image_type") or kind
            _log(f"    [{page_idx}/{total_pages}] {page_data.source_file[:50]:<50} p.{page_data.page_number:<3} -> {str(label)[:25]:<25} ({page_dt_ms} ms)")

            reviewer_flags = _default_reviewer_flags(page_data)
            case_result.pages.append(page_data)
            case_result.reviewer_flags.extend(reviewer_flags)

        # Case-level fusion
        fusion_t0 = _time.time()
        try:
            _apply_case_fusion(case_result)
            fusion_dt_ms = int((_time.time() - fusion_t0) * 1000)
            _log(f"    fusion: pre/post correlation + STG alignment ({fusion_dt_ms} ms)")
        except Exception as e:
            _log(f"    fusion FAILED: {e}")

        # Build the structured row + flush this package's JSON to disk NOW.
        try:
            structured_row = build_package_json_row(case_result)
            case_result.structured_output_row = structured_row
            case_result.textual_summary = _extract_summary_pdf(structured_row)
            case_result.final_case_summary = _default_case_level_summary(case_result)

            run_outputs.all_case_results.append(case_result)
            run_outputs.package_json_rows.setdefault(pkg, []).append(structured_row)
            run_outputs.final_reviewer_flags.extend(case_result.reviewer_flags)
            run_outputs.final_case_summaries.append(case_result.final_case_summary)

            written = _write_package_partial(OUTPUT_ROOT, pkg, run_outputs.package_json_rows[pkg])
            case_dt_s = _time.time() - case_t0
            _log(f"  done in {case_dt_s:.1f}s · {len(case_result.reviewer_flags)} flag(s) · wrote {written.name} ({len(run_outputs.package_json_rows[pkg])} rows)\n")
        except Exception as e:
            case_dt_s = _time.time() - case_t0
            _log(f"  EXPORT FAILED ({case_dt_s:.1f}s): {e}\n")

    pipeline_dt_s = _time.time() - pipeline_t0
    _log("=" * 60)
    _log(f"pipeline finished in {pipeline_dt_s:.1f}s")
    _log(f"total cases : {len(run_outputs.all_case_results)}")
    _log(f"total flags : {len(run_outputs.final_reviewer_flags)}")
    for pkg_code, rows in sorted(run_outputs.package_json_rows.items()):
        _log(f"  {pkg_code}: {len(rows)} rows")
    _log("=" * 60)

    return run_outputs


In [16]:

# =========================
# DISPLAY / FINAL OUTPUT SECTION
# =========================

def display_final_outputs(run_outputs: RunOutputs) -> None:
    """
    Display final results at the very end of the notebook.
    """

    print("=" * 80)
    print("PS2 PIPELINE COMPLETE — PACKAGE-SPECIFIC JSON OUTPUTS")
    print("=" * 80)
    print(f"Cases processed: {len(run_outputs.all_case_results)}")
    print(f"Reviewer flags: {len(run_outputs.final_reviewer_flags)}")
    print(f"Case summaries: {len(run_outputs.final_case_summaries)}")
    if run_outputs.written_files:
        print("Written files:")
        print(json.dumps(ensure_json_serializable(run_outputs.written_files), indent=2, ensure_ascii=False))

    for package_code in sorted(run_outputs.package_json_rows.keys()):
        rows = run_outputs.package_json_rows.get(package_code, [])
        print("\n" + "=" * 80)
        print(f"PACKAGE {package_code} — STRUCTURED JSON ROWS")
        print("=" * 80)
        if not rows:
            print("No rows available.")
        else:
            print(json.dumps(ensure_json_serializable(rows), indent=2, ensure_ascii=False))

    print("\n" + "=" * 80)
    print("FINAL REVIEWER FLAGS")
    print("=" * 80)
    print(json.dumps(ensure_json_serializable(run_outputs.final_reviewer_flags), indent=2, ensure_ascii=False))

    print("\n" + "=" * 80)
    print("FINAL CASE SUMMARIES")
    print("=" * 80)
    print(json.dumps(ensure_json_serializable(run_outputs.final_case_summaries), indent=2, ensure_ascii=False))


In [ ]:

# =========================
# EXECUTION CELL
# =========================

# Participants are expected to implement the pipeline stage functions above.
# Once implemented, the following is the intended end-of-notebook flow.

run_outputs = run_ps2_pipeline(INPUT_ROOT)
write_run_outputs(run_outputs, OUTPUT_ROOT)
display_final_outputs(run_outputs)



PS2 PIPELINE START  (provider=nha, input=Claims)
output: G:\PROJECTS\medical_reports_summary\outputs
discovered 94 files across 40 cases

[1/40] MC011A/BOCW_GJ_R3_2026040310046613_ER  (4 files, 24 pages)


2026-05-01 01:25:33,176 - INFO - Fetching new auth token...
2026-05-01 01:25:33,335 - INFO - HTTP Request: POST https://aaehackathon.nhaad.in/controlplane/iudx/v2/auth/token "HTTP/1.1 200 OK"
2026-05-01 01:25:33,336 - INFO - Auth token obtained successfully
2026-05-01 01:25:37,549 - INFO - HTTP Request: POST https://aaehackathon.nhaad.in/nhm-proxy/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-01 01:25:39,955 - INFO - HTTP Request: POST https://aaehackathon.nhaad.in/nhm-proxy/v1/chat/completions "HTTP/1.1 200 OK"


    [1/24] 000009__BOCW_GJ_R3_2026040310046613__7a877f13-90de p.1   -> Coronary Angiogram        (6831 ms)


2026-05-01 01:25:43,399 - INFO - HTTP Request: POST https://aaehackathon.nhaad.in/nhm-proxy/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-01 01:25:45,156 - INFO - HTTP Request: POST https://aaehackathon.nhaad.in/nhm-proxy/v1/chat/completions "HTTP/1.1 200 OK"


    [2/24] 000009__BOCW_GJ_R3_2026040310046613__7a877f13-90de p.2   -> Coronary Angiogram        (5200 ms)


...

2026-05-01 01:32:57,890 - INFO - HTTP Request: POST https://aaehackathon.nhaad.in/nhm-proxy/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-01 01:32:58,619 - INFO - HTTP Request: POST https://aaehackathon.nhaad.in/nhm-proxy/v1/chat/completions "HTTP/1.1 200 OK"


    fusion: pre/post correlation + STG alignment (1492 ms)
  done in 84.7s · 0 flag(s) · wrote MC011A_outputs.json (5 rows)

[6/40] MC011A/PMJAY_GJ_S_G_R3_2026040410001743  (4 files, 4 pages)


2026-05-01 01:33:06,969 - INFO - HTTP Request: POST https://aaehackathon.nhaad.in/nhm-proxy/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-01 01:33:08,569 - INFO - HTTP Request: POST https://aaehackathon.nhaad.in/nhm-proxy/v1/chat/completions "HTTP/1.1 200 OK"


    [1/4] 000160__PMJAY_GJ_S_G_R3_2026040410001743__PTCA_IMA p.1   -> Coronary Angiogram        (9948 ms)


2026-05-01 01:33:21,418 - INFO - HTTP Request: POST https://aaehackathon.nhaad.in/nhm-proxy/v1/chat/completions "HTTP/1.1 200 OK"
